# Table of Contents

1. [Environment Setup](#1-environment-setup)
2. [Configuration](#2-configuration)
3. [Reproducibility and Device Setup](#3-reproducibility-and-device-setup)
4. [Dataset Discovery and Metadata Cache](#4-dataset-discovery-and-metadata-cache)
5. [Dependencies and Imports](#5-dependencies-and-imports)
6. [Data Loading and Preprocessing](#6-data-loading-and-preprocessing)
7. [Model Architecture](#7-model-architecture)
8. [Experiment Tracking](#8-experiment-tracking)
9. [Training Utilities](#9-training-utilities)
10. [Training Loop](#10-training-loop)
11. [Evaluation](#11-evaluation) (includes threshold sweep, pixel AUC, confusion matrix, forgery-type, mask-size, shortcut checks)
12. [Visualization of Predictions](#12-visualization-of-predictions)
13. [Advanced Analysis](#13-advanced-analysis) (failure cases)
14. [Explainability](#14-explainability) (Grad-CAM)
15. [Robustness Testing](#15-robustness-testing)
16. [Inference Examples](#16-inference-examples)
17. [Conclusion](#17-conclusion)

# Tampered Image Detection and Localization (vK.10.6)

This Kaggle-first notebook presents a complete assignment submission for tampered image
detection and tampered region localization.

**Engineering upgrades in vK.10.6**
- **Advanced evaluation suite:** threshold sweep, pixel-level AUC, forgery-type breakdown, mask-size stratification
- **Explainability:** Grad-CAM heatmaps on encoder bottleneck
- **Robustness testing:** 8 degradation conditions (JPEG, noise, blur, resize) with delta from clean baseline
- **Shortcut learning validation:** mask randomization and boundary sensitivity tests
- **Failure case analysis:** 10 worst predictions with metadata annotation
- **Data leakage verification:** path overlap assertion
- **Confusion matrix, ROC curve, PR curve** with threshold-optimized metrics

**Carried forward from vK.10.5**
- Multi-GPU training with `nn.DataParallel` (utilizes both Kaggle T4 GPUs)
- Centralized CONFIG dictionary for all hyperparameters
- Full reproducibility seeding (Python, NumPy, PyTorch CPU/CUDA)
- Automatic Mixed Precision (AMP) for faster training
- Three-file checkpoint system with automatic resume
- Early stopping based on tampered-only Dice coefficient
- Tampered-only metric reporting to address metric inflation
- GPU diagnostics and VRAM-based batch size auto-adjustment
- Metadata caching to skip redundant dataset scanning
- Optimized DataLoaders (persistent workers, seeded workers, drop_last)

**Notebook deliverables**
- Image-level tamper detection through the classifier head
- Pixel-level tampered region localization through the segmentation branch
- Reproducible Kaggle-first execution with Colab/Drive fallback
- Qualitative visual evidence showing predicted masks and overlays
- Comprehensive 12-point evaluation suite with robustness testing

## Project Objectives: Fulfilled vs Remaining

| Requirement | Status | Evidence |
|---|---|---|
| Dataset: authentic + tampered + masks | Fulfilled | CASIA dataset with IMAGE/MASK dirs |
| Model performs detection + localization | Fulfilled | UNetWithClassifier dual-head |
| Evaluation with Dice / IoU / F1 | Fulfilled | Tampered-only and all-sample metrics |
| Visual results (Original, GT, Pred, Overlay) | Fulfilled | Submission prediction grid |
| Single notebook | Fulfilled | All code in one notebook |
| Reproducibility | Fulfilled | Full seeding + checkpoint resume |
| AMP training | Fulfilled | autocast + GradScaler |
| Early stopping | Fulfilled | Patience-based on tampered Dice |
| Multi-GPU utilization | Fulfilled | nn.DataParallel across T4 ×2 |
| Threshold optimization | Fulfilled | Sweep 0.05-0.80, select best F1 |
| Robustness testing | Fulfilled | 8 degradation conditions with deltas |
| Grad-CAM explainability | Fulfilled | Encoder bottleneck heatmaps |
| Confusion matrix + ROC/PR | Fulfilled | seaborn heatmap + sklearn curves |
| Forgery-type breakdown | Fulfilled | Splicing vs copy-move metrics |
| Data leakage verification | Fulfilled | Path overlap assertion |
| Shortcut learning checks | Fulfilled | Mask randomization + boundary sensitivity |
| Failure case analysis | Fulfilled | 10 worst predictions annotated |
| Pixel-level AUC-ROC | Fulfilled | Threshold-independent localization metric |

## 1. Environment Setup

Installs dependencies, defines dataset paths, and locates the dataset using a three-tier
fallback: Kaggle attached dataset → Google Drive → Kaggle API download.

In [1]:
import subprocess
import sys
import os
import shutil
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
KAGGLE_DATASET_SLUG = "harshv777/casia2-0-upgraded-dataset"
KAGGLE_INPUT_DIR = Path("/kaggle/input")
KAGGLE_WORKING_DIR = Path("/kaggle/working")
ATTACHED_DATASET_DIR = KAGGLE_INPUT_DIR / "casia2-0-upgraded-dataset"
DRIVE_SEARCH_ROOTS = [Path("/content/drive/MyDrive"), Path("/content/drive/Shareddrives")]

if IN_COLAB:
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q",
        "albumentations==1.3.1", "opencv-python-headless==4.10.0.84", "kaggle",
    ])

KAGGLE_WORKING_DIR.mkdir(parents=True, exist_ok=True)

print("IN_COLAB:", IN_COLAB)
print("KAGGLE_INPUT_DIR:", KAGGLE_INPUT_DIR)
print("KAGGLE_WORKING_DIR:", KAGGLE_WORKING_DIR)
print("ATTACHED_DATASET_DIR:", ATTACHED_DATASET_DIR)


IN_COLAB: False
KAGGLE_INPUT_DIR: /kaggle/input
KAGGLE_WORKING_DIR: /kaggle/working
ATTACHED_DATASET_DIR: /kaggle/input/casia2-0-upgraded-dataset


In [2]:
def has_valid_layout(path):
    """Check for IMAGE/ and MASK/ subdirectories."""
    if not path or not path.is_dir():
        return False
    children = {c.name.lower() for c in path.iterdir() if c.is_dir()}
    return "image" in children and "mask" in children


def find_in_kaggle_input(input_dir):
    """Search /kaggle/input/ for any attached dataset with IMAGE+MASK layout."""
    if input_dir is None or not input_dir.exists() or not input_dir.is_dir():
        return None
    preferred, fallback = [], []
    # Exact slug match
    exact = input_dir / "casia2-0-upgraded-dataset"
    if exact.exists() and has_valid_layout(exact):
        preferred.append(exact)
    # Direct children
    for child in sorted(input_dir.iterdir()):
        if child.is_dir() and has_valid_layout(child):
            fallback.append(child)
    # Recursive scan (Kaggle may nest under /kaggle/input/datasets/user/slug/)
    for d in sorted(input_dir.rglob("*")):
        if d.is_dir() and has_valid_layout(d):
            if "casia" in d.name.lower():
                preferred.append(d)
            else:
                fallback.append(d)
    candidates = preferred or fallback
    return candidates[0] if candidates else None


def find_in_drive(roots):
    """Search Google Drive mount points for the dataset folder."""
    for root in roots:
        if not root.exists():
            continue
        candidate = root / "casia2-0-upgraded-dataset"
        if candidate.exists() and has_valid_layout(candidate):
            return candidate
        for match in root.rglob("*casia*"):
            if match.is_dir() and has_valid_layout(match):
                return match
    return None


def download_from_kaggle(slug, target_dir):
    """Download dataset via Kaggle API to /kaggle/working/ (writable)."""
    try:
        from kaggle.api.kaggle_api_extended import KaggleApi
        api = KaggleApi()
        api.authenticate()
        # Download to writable directory, NOT /kaggle/input/ (read-only on Kaggle)
        download_dir = KAGGLE_WORKING_DIR / "_dataset_download"
        download_dir.mkdir(parents=True, exist_ok=True)
        api.dataset_download_files(slug, path=str(download_dir), unzip=True, quiet=False)
        if has_valid_layout(download_dir):
            return download_dir
        for d in download_dir.rglob("*"):
            if d.is_dir() and has_valid_layout(d):
                return d
    except Exception as e:
        print(f"Kaggle API fallback failed: {e}")
    return None


In [3]:
# DEBUG: show what Kaggle attached under /kaggle/input/
if KAGGLE_INPUT_DIR.exists():
    print("Contents of", KAGGLE_INPUT_DIR)
    for p in sorted(KAGGLE_INPUT_DIR.iterdir()):
        print(f"  {p.name}/  ->  {[c.name for c in p.iterdir() if c.is_dir()][:10] if p.is_dir() else 'file'}")
else:
    print(f"{KAGGLE_INPUT_DIR} does not exist")

dataset_dir = None

# 1) Kaggle attached dataset (default) — scan all attached datasets
dataset_dir = find_in_kaggle_input(KAGGLE_INPUT_DIR)
if dataset_dir:
    print(f"Using attached Kaggle dataset: {dataset_dir}")

# 2) Google Drive fallback (Colab only)
if dataset_dir is None and IN_COLAB:
    try:
        from google.colab import drive
        drive.mount("/content/drive", force_remount=False)
        found = find_in_drive(DRIVE_SEARCH_ROOTS)
        if found:
            dataset_dir = found
            print(f"Using Google Drive dataset: {dataset_dir}")
    except Exception as e:
        print(f"Google Drive fallback skipped: {e}")

# 3) Kaggle API download (last resort — writes to /kaggle/working/)
if dataset_dir is None:
    dataset_dir = download_from_kaggle(KAGGLE_DATASET_SLUG, KAGGLE_WORKING_DIR)
    if dataset_dir:
        print(f"Downloaded dataset via Kaggle API: {dataset_dir}")

if not dataset_dir or not has_valid_layout(dataset_dir):
    raise FileNotFoundError(
        "Dataset not found. Attach harshv777/casia2-0-upgraded-dataset on Kaggle, "
        "or place it in Google Drive, or configure Kaggle API credentials."
    )

print(f"Dataset root: {dataset_dir}")
print(f"  IMAGE dir: {(dataset_dir / 'IMAGE').exists()}")
print(f"  MASK dir:  {(dataset_dir / 'MASK').exists()}")


Contents of /kaggle/input
  datasets/  ->  ['harshv777']
Using attached Kaggle dataset: /kaggle/input/datasets/harshv777/casia2-0-upgraded-dataset
Dataset root: /kaggle/input/datasets/harshv777/casia2-0-upgraded-dataset
  IMAGE dir: True
  MASK dir:  True


In [4]:
import os
devnull = os.open(os.devnull, os.O_WRONLY)
os.dup2(devnull, 2)  # redirect C-level stderr to /dev/null

2

## 2. Configuration

All hyperparameters, feature flags, and path settings are centralized in a single
`CONFIG` dictionary.  This replaces the scattered constants from previous notebook
versions and makes experiment iteration easier.  Every training, evaluation, and
data-loading cell reads from `CONFIG` instead of defining its own local constants.


In [5]:
import os
from pathlib import Path

SEED = 42

CONFIG = {
    # -- Data --
    'image_size': 256,
    'batch_size': 8,             # auto-adjusted based on GPU VRAM
    'num_workers': 4,
    'train_ratio': 0.70,

    # -- Model --
    'architecture': 'UNetWithClassifier',
    'n_channels': 3,
    'n_classes': 1,
    'n_labels': 2,
    'dropout': 0.5,

    # -- Optimizer --
    'learning_rate': 1e-4,
    'weight_decay': 1e-4,
    'max_grad_norm': 5.0,

    # -- Scheduler --
    'scheduler': 'CosineAnnealingLR',
    'scheduler_T_max': 50,            # single cosine decay over all epochs

    # -- Loss --
    'alpha': 1.5,                # classification loss weight
    'beta': 1.0,                 # segmentation loss weight
    'focal_gamma': 2.0,
    'seg_bce_weight': 0.5,
    'seg_dice_weight': 0.5,

    # -- Training --
    'max_epochs': 100,
    'patience': 30,              # early stopping patience
    'checkpoint_every': 10,      # periodic checkpoint interval

    # -- Feature Flags --
    'use_amp': True,
    'use_wandb': True,
    'seg_threshold': 0.5,

    # -- Reproducibility --
    'seed': SEED,
}

# -- Output Directories --
CHECKPOINT_DIR = Path(KAGGLE_WORKING_DIR) / 'checkpoints'
RESULTS_DIR    = Path(KAGGLE_WORKING_DIR) / 'results'
PLOTS_DIR      = Path(KAGGLE_WORKING_DIR) / 'plots'

for d in [CHECKPOINT_DIR, RESULTS_DIR, PLOTS_DIR]:
    os.makedirs(d, exist_ok=True)

print('CONFIG:')
for k, v in CONFIG.items():
    print(f'  {k}: {v}')
print(f'\nCHECKPOINT_DIR: {CHECKPOINT_DIR}')
print(f'RESULTS_DIR:    {RESULTS_DIR}')
print(f'PLOTS_DIR:      {PLOTS_DIR}')

CONFIG:
  image_size: 256
  batch_size: 8
  num_workers: 4
  train_ratio: 0.7
  architecture: UNetWithClassifier
  n_channels: 3
  n_classes: 1
  n_labels: 2
  dropout: 0.5
  learning_rate: 0.0001
  weight_decay: 0.0001
  max_grad_norm: 5.0
  scheduler: CosineAnnealingLR
  scheduler_T_max: 50
  alpha: 1.5
  beta: 1.0
  focal_gamma: 2.0
  seg_bce_weight: 0.5
  seg_dice_weight: 0.5
  max_epochs: 100
  patience: 30
  checkpoint_every: 10
  use_amp: True
  use_wandb: True
  seg_threshold: 0.5
  seed: 42

CHECKPOINT_DIR: /kaggle/working/checkpoints
RESULTS_DIR:    /kaggle/working/results
PLOTS_DIR:      /kaggle/working/plots


### 2.1 Hyperparameter Summary

| Hyperparameter | Value | Source |
|---|---|---|
| `image_size` | 256 | Preserved from vK.7.1 |
| `batch_size` | 8 (auto-adjusted) | VRAM-based |
| `learning_rate` | 1e-4 | Preserved |
| `max_epochs` | 50 | Preserved |
| `alpha` (cls weight) | 1.5 | Preserved |
| `beta` (seg weight) | 1.0 | Preserved |
| `focal_gamma` | 2.0 | Preserved |
| `scheduler` | CosineAnnealingLR(T_max=10) | Preserved |
| `patience` | 10 | New (early stopping) |
| `use_amp` | True | New (mixed precision) |


## 3. Reproducibility and Device Setup

Full reproducibility is enforced through seeded random number generators across
Python, NumPy, and PyTorch.  GPU diagnostics confirm hardware capabilities and
auto-adjust the batch size based on available VRAM.


In [6]:
import random
import numpy as np
import torch


def set_seed(seed):
    """Set seeds for full reproducibility across all libraries."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


set_seed(CONFIG['seed'])

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

if torch.cuda.is_available():
    n_gpus = torch.cuda.device_count()

    for gpu_id in range(n_gpus):
        gpu_name = torch.cuda.get_device_name(gpu_id)
        vram_gb = torch.cuda.get_device_properties(gpu_id).total_memory / 1e9
        print(f'GPU {gpu_id}:          {gpu_name} ({vram_gb:.1f} GB)')

    total_vram_gb = sum(torch.cuda.get_device_properties(i).total_memory for i in range(n_gpus)) / 1e9

    # TF32 for faster matmul on Ampere+ (does not affect determinism)
    if hasattr(torch.backends.cuda.matmul, 'fp32_precision'):
        torch.backends.cuda.matmul.fp32_precision = 'tf32'
        torch.backends.cudnn.conv.fp32_precision = 'tf32'
    else:
        torch.backends.cuda.matmul.allow_tf32 = True
        torch.backends.cudnn.allow_tf32 = True

    print(f'GPUs available: {n_gpus}')
    print(f'Total VRAM:     {total_vram_gb:.1f} GB')
    print(f'cuDNN:          {torch.backends.cudnn.enabled}')
    print(f'Deterministic:  {torch.backends.cudnn.deterministic}')
    print(f'AMP:            {"enabled" if CONFIG["use_amp"] else "disabled"}')

    # Auto-adjust batch size based on total available VRAM across all GPUs
    if total_vram_gb >= 28:
        CONFIG['batch_size'] = 32
    elif total_vram_gb >= 20:
        CONFIG['batch_size'] = 24
    elif total_vram_gb >= 15:
        CONFIG['batch_size'] = 16
    # else keep CONFIG default (8)
    print(f'Batch size (auto-adjusted for {n_gpus} GPU{"s" if n_gpus > 1 else ""}): {CONFIG["batch_size"]}')
else:
    n_gpus = 0
    print('WARNING: No GPU detected. Training will be extremely slow.')

print(f'Device: {device}')

GPU 0:          Tesla T4 (15.6 GB)
GPU 1:          Tesla T4 (15.6 GB)
GPUs available: 2
Total VRAM:     31.3 GB
cuDNN:          True
Deterministic:  True
AMP:            enabled
Batch size (auto-adjusted for 2 GPUs): 32
Device: cuda


The batch size is auto-adjusted based on available GPU VRAM:
- **>=15 GB** (Kaggle P100 / Colab T4): `batch_size=32`
- **>=10 GB**: `batch_size=16`
- **<10 GB**: `batch_size=8` (default)

The effective hyperparameters logged to W&B reflect this adjusted value.


## 4. Dataset Discovery and Metadata Cache

The notebook operates on an image tampering dataset organized into authentic and
tampered images with corresponding binary ground truth masks.

**Metadata caching:** If a valid `image_mask_metadata.csv` already exists with the
expected row count, the scanning step is skipped entirely to reduce startup time.


#### 2.1.1 Locate IMAGE and MASK Directories

Scans the normalized Kaggle-style input path and searches recursively for the `IMAGE` and `MASK` folders.

In [7]:
import os
from pathlib import Path
import pandas as pd

# =========================
# 1) Discover the dataset root
# =========================

# Inspect the Kaggle input directory so the notebook can confirm the expected dataset is available.
INPUT_ROOT = KAGGLE_INPUT_DIR

print(f"Available datasets under {INPUT_ROOT}:")
for p in INPUT_ROOT.iterdir():
    if p.is_dir():
        print(" -", p.name)

# Use the Kaggle-first attached dataset path prepared in the environment setup cells.
DATASET_DIR = dataset_dir  # resolved in Environment Setup (Kaggle -> Drive -> API)

# Search recursively for IMAGE and MASK so the metadata build still works if the dataset is nested one level deeper.
IMAGE_DIR = None
MASK_DIR = None

for p in DATASET_DIR.rglob("*"):
    if p.is_dir() and p.name.lower() == "image":
        IMAGE_DIR = p
    if p.is_dir() and p.name.lower() == "mask":
        MASK_DIR = p

print("IMAGE_DIR:", IMAGE_DIR)
print("MASK_DIR:", MASK_DIR)

assert IMAGE_DIR is not None, "Could not find the IMAGE directory. Verify the dataset folder name and structure."
assert MASK_DIR is not None, "Could not find the MASK directory. Verify the dataset folder name and structure."

Available datasets under /kaggle/input:
 - datasets
IMAGE_DIR: /kaggle/input/datasets/harshv777/casia2-0-upgraded-dataset/IMAGE
MASK_DIR: /kaggle/input/datasets/harshv777/casia2-0-upgraded-dataset/MASK


#### 2.1.2 Build Metadata Table from Image-Mask Pairs

Iterates over authentic (`Au`) and tampered (`Tp`) subdirectories,
pairing each image with its corresponding mask and recording the image-level label.


In [8]:
# =========================
# Build metadata with caching
# =========================

output_csv = KAGGLE_WORKING_DIR / "image_mask_metadata.csv"

# Count images in dataset to validate cache
_au_count = sum(1 for f in (IMAGE_DIR / 'Au').iterdir() if f.is_file())
_tp_count = sum(1 for f in (IMAGE_DIR / 'Tp').iterdir() if f.is_file())
_expected_count = _au_count + _tp_count

df = None
if output_csv.exists():
    cached_df = pd.read_csv(output_csv)
    if len(cached_df) == _expected_count:
        print(f'Using cached metadata ({len(cached_df)} rows): {output_csv}')
        df = cached_df
    else:
        print(f'Cache stale ({len(cached_df)} rows vs {_expected_count} expected), rebuilding...')

if df is None:
    label_folders = {
        "Au": {"class_name": "authentic", "label": 0},
        "Tp": {"class_name": "tampered",  "label": 1},
    }
    valid_exts = {".jpg", ".jpeg", ".png", ".tif", ".tiff", ".bmp"}
    rows = []

    for sub_name, info in label_folders.items():
        img_subdir = IMAGE_DIR / sub_name
        mask_subdir = MASK_DIR / sub_name
        if not img_subdir.exists():
            print(f'Warning: image folder does not exist: {img_subdir}')
            continue
        print(f'Scanning images in: {img_subdir}')
        for img_path in img_subdir.iterdir():
            if not img_path.is_file():
                continue
            if img_path.suffix.lower() not in valid_exts:
                continue
            mask_path = mask_subdir / img_path.name
            mask_exists = mask_path.exists()
            rows.append({
                "image_path": str(img_path),
                "mask_path": str(mask_path) if mask_exists else "",
                "mask_exists": int(mask_exists),
                "class_folder": sub_name,
                "class_name": info["class_name"],
                "label": info["label"],
            })

    print(f'Total images found: {len(rows)}')
    df = pd.DataFrame(rows)
    df.to_csv(output_csv, index=False)
    print(f'CSV saved to: {output_csv}')

print(df.head())
print('\nCounts per class_name:')
print(df['class_name'].value_counts())
print('\nMissing masks:')
print(df[df['mask_exists'] == 0].head())

Scanning images in: /kaggle/input/datasets/harshv777/casia2-0-upgraded-dataset/IMAGE/Au
Scanning images in: /kaggle/input/datasets/harshv777/casia2-0-upgraded-dataset/IMAGE/Tp
Total images found: 12614
CSV saved to: /kaggle/working/image_mask_metadata.csv
                                          image_path  \
0  /kaggle/input/datasets/harshv777/casia2-0-upgr...   
1  /kaggle/input/datasets/harshv777/casia2-0-upgr...   
2  /kaggle/input/datasets/harshv777/casia2-0-upgr...   
3  /kaggle/input/datasets/harshv777/casia2-0-upgr...   
4  /kaggle/input/datasets/harshv777/casia2-0-upgr...   

                                           mask_path  mask_exists  \
0  /kaggle/input/datasets/harshv777/casia2-0-upgr...            1   
1  /kaggle/input/datasets/harshv777/casia2-0-upgr...            1   
2  /kaggle/input/datasets/harshv777/casia2-0-upgr...            1   
3  /kaggle/input/datasets/harshv777/casia2-0-upgr...            1   
4  /kaggle/input/datasets/harshv777/casia2-0-upgr...          

### 2.2 Train / Validation / Test Split

Loads the metadata CSV, applies a stratified 70/15/15 split to preserve class balance
across train, validation, and test subsets, and saves each split to a separate CSV file.


In [9]:
import pandas as pd
from sklearn.model_selection import train_test_split

train_csv = KAGGLE_WORKING_DIR / 'train_metadata.csv'
val_csv   = KAGGLE_WORKING_DIR / 'val_metadata.csv'
test_csv  = KAGGLE_WORKING_DIR / 'test_metadata.csv'

# Check for cached splits
if train_csv.exists() and val_csv.exists() and test_csv.exists():
    train_df = pd.read_csv(train_csv)
    val_df   = pd.read_csv(val_csv)
    test_df  = pd.read_csv(test_csv)
    if len(train_df) + len(val_df) + len(test_df) == len(df):
        print(f'Using cached splits: train={len(train_df)}, val={len(val_df)}, test={len(test_df)}')
    else:
        print('Cached splits stale, rebuilding...')
        train_df = None
else:
    train_df = None

if train_df is None:
    train_df, temp_df = train_test_split(
        df, test_size=0.30, stratify=df['label'], random_state=CONFIG['seed'],
    )
    val_df, test_df = train_test_split(
        temp_df, test_size=0.50, stratify=temp_df['label'], random_state=CONFIG['seed'],
    )
    train_df.to_csv(train_csv, index=False)
    val_df.to_csv(val_csv, index=False)
    test_df.to_csv(test_csv, index=False)
    print(f'Splits saved: train={len(train_df)}, val={len(val_df)}, test={len(test_df)}')

print(f'\nTrain class distribution:')
print(train_df['class_name'].value_counts())
print(f'\nVal class distribution:')
print(val_df['class_name'].value_counts())
print(f'\nTest class distribution:')
print(test_df['class_name'].value_counts())

Splits saved: train=8829, val=1892, test=1893

Train class distribution:
class_name
authentic    5243
tampered     3586
Name: count, dtype: int64

Val class distribution:
class_name
authentic    1124
tampered      768
Name: count, dtype: int64

Test class distribution:
class_name
authentic    1124
tampered      769
Name: count, dtype: int64


### 4.4 Dataset Summary

Quick summary of dataset splits and class balance.

In [10]:
print(f"{'Split':<8} {'Total':>6} {'Authentic':>10} {'Tampered':>9}")
print('-' * 38)
for name, split_df in [('Train', train_df), ('Val', val_df), ('Test', test_df)]:
    n_auth = (split_df['label'] == 0).sum()
    n_tamp = (split_df['label'] == 1).sum()
    print(f'{name:<8} {len(split_df):>6} {n_auth:>10} {n_tamp:>9}')

Split     Total  Authentic  Tampered
--------------------------------------
Train      8829       5243      3586
Val        1892       1124       768
Test       1893       1124       769


### 4.5 Data Leakage Verification

Verify zero overlap between train/val/test splits at the path level. This catches silent bugs where images appear in multiple splits.

In [11]:
# ================== Data Leakage Verification ==================
train_paths = set(train_df['image_path'].tolist())
val_paths   = set(val_df['image_path'].tolist())
test_paths  = set(test_df['image_path'].tolist())

tv_overlap = train_paths & val_paths
tt_overlap = train_paths & test_paths
vt_overlap = val_paths & test_paths

assert len(tv_overlap) == 0, f"LEAK: {len(tv_overlap)} images in both train and val!"
assert len(tt_overlap) == 0, f"LEAK: {len(tt_overlap)} images in both train and test!"
assert len(vt_overlap) == 0, f"LEAK: {len(vt_overlap)} images in both val and test!"

total_unique = len(train_paths | val_paths | test_paths)
total_rows = len(train_df) + len(val_df) + len(test_df)
assert total_unique == total_rows, f"LEAK: {total_rows} rows but only {total_unique} unique paths!"

print("DATA LEAKAGE CHECK: PASS")
print(f"  Train: {len(train_paths)} | Val: {len(val_paths)} | Test: {len(test_paths)}")
print(f"  Total unique paths: {total_unique} == Total rows: {total_rows}")

DATA LEAKAGE CHECK: PASS
  Train: 8829 | Val: 1892 | Test: 1893
  Total unique paths: 12614 == Total rows: 12614


## 5. Dependencies and Imports

All training, evaluation, and visualization imports are consolidated here
to avoid redundant imports scattered across multiple cells.


In [12]:
import cv2
import sys
import math
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import roc_auc_score

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.amp import autocast, GradScaler

from tqdm.auto import tqdm

import albumentations as A
from albumentations.pytorch import ToTensorV2

import matplotlib
matplotlib.use('Agg')  # non-interactive backend for Kaggle
import matplotlib.pyplot as plt

print('Imports complete.')
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')

Imports complete.
PyTorch: 2.9.0+cu126
CUDA available: True


## 6. Data Loading and Preprocessing

The source notebook loads metadata CSV files, builds PyTorch datasets with
Albumentations augmentation, and creates dataloaders for training, validation,
and test splits.


### 6.1 Image and Mask Transforms

Defines Albumentations pipelines for the training split (with augmentation) and the
validation/test splits (resize and normalization only).


In [13]:
# ================== Define transforms ==================
IMAGE_SIZE = CONFIG['image_size']

def get_train_transform():
    """
    Purpose:
        Build the augmentation pipeline used for training images and masks.
    """
    return A.Compose([
        A.Resize(IMAGE_SIZE, IMAGE_SIZE),
        A.HorizontalFlip(p=0.5),
        A.RandomBrightnessContrast(p=0.3),
        A.GaussNoise(p=0.25),
        A.ImageCompression(quality_range=(50, 90), p=0.25),
        A.Affine(
            translate_percent={'x': (-0.02, 0.02), 'y': (-0.02, 0.02)},
            scale=(0.9, 1.1),
            rotate=(-10, 10),
            border_mode=cv2.BORDER_REFLECT_101,
            p=0.5,
        ),
        A.Normalize(mean=(0.485, 0.456, 0.406),
                    std=(0.229, 0.224, 0.225)),
        ToTensorV2(),
    ])

def get_valid_transform():
    """
    Purpose:
        Build the deterministic preprocessing pipeline for validation and test samples.
    """
    return A.Compose([
        A.Resize(IMAGE_SIZE, IMAGE_SIZE),
        A.Normalize(mean=(0.485, 0.456, 0.406),
                    std=(0.229, 0.224, 0.225)),
        ToTensorV2(),
    ])

### 6.2 Dataset Class

The `ImageMaskDataset` class loads image-mask pairs from metadata, applies shared transforms,
and returns tensors compatible with the dual-head model.


In [14]:
# ================== 3) Define the dataset ==================
class ImageMaskDataset(Dataset):
    """
    Purpose:
        Load paired images, masks, and image-level tamper labels from the metadata CSV files.

    Key Responsibilities:
        - Read each image and its corresponding mask from disk.
        - Apply shared preprocessing so classification and localization are trained on aligned inputs.

    Notes:
        The dataset returns image tensor, binary mask tensor, and image-level label.
    """
    def __init__(self, df, transform=None):
        """
        Purpose:
            Store metadata and an optional transform pipeline for later sample loading.

        Inputs:
            df (pd.DataFrame): Metadata table with image paths, mask paths, and labels.
            transform (callable): Optional Albumentations transform applied to image and mask.

        Returns:
            None.

        Notes:
            The dataframe index is reset to preserve stable integer access during training.
        """
        self.df = df.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        """
        Purpose:
            Return the number of examples in the dataset.

        Inputs:
            None.

        Returns:
            int: Number of metadata rows.

        Notes:
            DataLoader uses this length to define epoch size.
        """
        return len(self.df)

    def __getitem__(self, idx):
        """
        Purpose:
            Load one image, its binary mask, and the image-level tamper label.

        Inputs:
            idx (int): Sample index inside the metadata table.

        Returns:
            tuple: Image tensor, mask tensor, and class label tensor.

        Notes:
            Binary masks make the segmentation target explicit: tampered pixels vs clean pixels.
        """
        row = self.df.iloc[idx]

        img_path = row["image_path"]
        mask_path = row["mask_path"]
        label = int(row["label"])

        image = cv2.imread(img_path)
        mask  = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)

        if image is None:
            raise RuntimeError(f"Failed to read image: {img_path}")
        if mask is None:
            # Authentic images may lack a physical mask file â use zeros
            mask = np.zeros((image.shape[0], image.shape[1]), dtype=np.uint8)

        # Convert the OpenCV image to RGB before normalization and visualization.
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        # Convert the grayscale mask to binary so the localization target has a clear foreground/background split.
        mask = (mask > 0).astype("float32")

        if self.transform is not None:
            # Apply identical spatial augmentation to the image and its supervision mask.
            augmented = self.transform(image=image, mask=mask)
            image = augmented["image"]
            mask  = augmented["mask"]
        else:
            image = torch.from_numpy(image).permute(2, 0, 1).float() / 255.0
            mask  = torch.from_numpy(mask).unsqueeze(0).float()

        if mask.ndim == 2:
            # Keep an explicit channel dimension for the segmentation branch and BCE-style losses.
            mask = mask.unsqueeze(0)

        return image, mask, torch.tensor(label, dtype=torch.long)


### 6.3 DataLoader Construction

Creates train, validation, and test DataLoaders with optimized settings:
- `persistent_workers` to avoid respawning workers each epoch
- `seed_worker` + `Generator` for reproducible data ordering
- `drop_last=True` for training to avoid uneven last-batch size
- `pin_memory=True` for faster GPU transfers


In [15]:
def seed_worker(worker_id):
    """Seed each DataLoader worker for reproducibility."""
    worker_seed = torch.initial_seed() % 2**32
    np.random.seed(worker_seed)
    random.seed(worker_seed)

g = torch.Generator()
g.manual_seed(CONFIG['seed'])

_nw = CONFIG['num_workers']
loader_kwargs = dict(
    num_workers=_nw,
    pin_memory=torch.cuda.is_available(),
    persistent_workers=_nw > 0,
)

train_dataset = ImageMaskDataset(train_df, transform=get_train_transform())
val_dataset   = ImageMaskDataset(val_df,   transform=get_valid_transform())
test_dataset  = ImageMaskDataset(test_df,  transform=get_valid_transform())

train_loader = DataLoader(
    train_dataset,
    batch_size=CONFIG['batch_size'],
    shuffle=True,
    drop_last=True,
    worker_init_fn=seed_worker,
    generator=g,
    **loader_kwargs,
)
val_loader = DataLoader(
    val_dataset,
    batch_size=CONFIG['batch_size'],
    shuffle=False,
    drop_last=False,
    **loader_kwargs,
)
test_loader = DataLoader(
    test_dataset,
    batch_size=CONFIG['batch_size'],
    shuffle=False,
    drop_last=False,
    **loader_kwargs,
)

print(f'Train: {len(train_dataset)} samples, {len(train_loader)} batches')
print(f'Val:   {len(val_dataset)} samples, {len(val_loader)} batches')
print(f'Test:  {len(test_dataset)} samples, {len(test_loader)} batches')
print(f'Workers: {_nw}, persistent: {loader_kwargs["persistent_workers"]}, batch_size: {CONFIG["batch_size"]}')

Train: 8829 samples, 275 batches
Val:   1892 samples, 60 batches
Test:  1893 samples, 60 batches
Workers: 4, persistent: True, batch_size: 32


### 6.4 Data Visualization

Visual sanity check before training: sample images with their masks and augmentations,
class distribution across splits, and image size statistics.

In [16]:
# ================== Pre-training data visualization ==================
import matplotlib.pyplot as plt
import numpy as np

def _denorm(img_t):
    """Reverse ImageNet normalization for display."""
    mean = np.array([0.485, 0.456, 0.406])
    std  = np.array([0.229, 0.224, 0.225])
    img = img_t.permute(1, 2, 0).numpy() * std + mean
    return np.clip(img, 0, 1)

# --- 1) Sample grid: 4 authentic + 4 tampered with masks ---
fig, axes = plt.subplots(4, 3, figsize=(10, 13))
fig.suptitle('Sample Images (top 2 authentic, bottom 2 tampered)', fontsize=13)

auth_indices = [i for i in range(len(train_dataset)) if train_df.iloc[i]['label'] == 0][:2]
tamp_indices = [i for i in range(len(train_dataset)) if train_df.iloc[i]['label'] == 1][:2]

for row, idx in enumerate(auth_indices + tamp_indices):
    img, mask, label = train_dataset[idx]
    lbl_str = 'Authentic' if label == 0 else 'Tampered'
    axes[row, 0].imshow(_denorm(img))
    axes[row, 0].set_title(f'{lbl_str} (idx {idx})')
    axes[row, 1].imshow(mask.squeeze().numpy(), cmap='gray', vmin=0, vmax=1)
    axes[row, 1].set_title('Ground Truth Mask')
    axes[row, 2].imshow(_denorm(img))
    axes[row, 2].imshow(mask.squeeze().numpy(), cmap='Reds', alpha=0.4)
    axes[row, 2].set_title('Overlay')
    for ax in axes[row]:
        ax.axis('off')
plt.tight_layout()
plt.show()

# --- 2) Class distribution per split ---
fig, axes = plt.subplots(1, 3, figsize=(12, 3))
for ax, (name, df) in zip(axes, [('Train', train_df), ('Val', val_df), ('Test', test_df)]):
    counts = df['label'].value_counts().sort_index()
    bars = ax.bar(['Authentic', 'Tampered'], [counts.get(0, 0), counts.get(1, 0)],
                  color=['#2ecc71', '#e74c3c'])
    ax.set_title(f'{name} (n={len(df)})')
    ax.set_ylabel('Count')
    for bar in bars:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
                str(int(bar.get_height())), ha='center', fontsize=9)
fig.suptitle('Class Distribution Across Splits', fontsize=13)
plt.tight_layout()
plt.show()

# --- 3) Augmentation preview: same image with 4 random augmentations ---
aug_idx = tamp_indices[0]  # pick first tampered sample
fig, axes = plt.subplots(1, 5, figsize=(15, 3))
fig.suptitle('Augmentation Preview (same tampered image, 4 random transforms)', fontsize=12)
raw_row = train_df.iloc[aug_idx]
raw_img = cv2.cvtColor(cv2.imread(raw_row['image_path']), cv2.COLOR_BGR2RGB)
raw_img_resized = cv2.resize(raw_img, (IMAGE_SIZE, IMAGE_SIZE))
axes[0].imshow(raw_img_resized)
axes[0].set_title('Original')
axes[0].axis('off')
aug_tf = get_train_transform()
raw_mask = cv2.imread(raw_row['mask_path'], cv2.IMREAD_GRAYSCALE) if pd.notna(raw_row.get('mask_path')) else np.zeros((raw_img.shape[0], raw_img.shape[1]), dtype=np.uint8)
for j in range(1, 5):
    augmented = aug_tf(image=raw_img, mask=raw_mask)
    aug_img = augmented['image']
    axes[j].imshow(_denorm(aug_img))
    axes[j].set_title(f'Aug {j}')
    axes[j].axis('off')
plt.tight_layout()
plt.show()

## 7. Model Architecture

A custom dual-head U-Net (`UNetWithClassifier`) that shares an encoder backbone and produces both **image-level classification** and **pixel-level segmentation** outputs.

```
                            Input (3 × 256 × 256)
                                    │
                          ┌─────────▼─────────┐
                          │   DoubleConv (64)  │  x1
                          └─────────┬─────────┘
                                    │ MaxPool2d
                          ┌─────────▼─────────┐
                          │    Down (128)      │  x2
                          └─────────┬─────────┘
                                    │ MaxPool2d
                          ┌─────────▼─────────┐
                          │    Down (256)      │  x3
                          └─────────┬─────────┘
                                    │ MaxPool2d
                          ┌─────────▼─────────┐
                          │    Down (512)      │  x4
                          └─────────┬─────────┘
                                    │ MaxPool2d
                          ┌─────────▼─────────┐
                          │  Bottleneck (1024) │  x5
                          └──┬─────────────┬───┘
                             │             │
           ┌─────────────────┘             └──────────────┐
           │ DECODER (Segmentation)             CLASSIFIER HEAD │
           │                                              │
  ┌────────▼────────┐                        AdaptiveAvgPool2d(1×1)
  │ Up(1024→512)+x4 │                                    │
  └────────┬────────┘                              Flatten
  ┌────────▼────────┐                                    │
  │ Up(512→256)+x3  │                          Linear(1024→512)
  └────────┬────────┘                             ReLU + Dropout(0.5)
  ┌────────▼────────┐                                    │
  │ Up(256→128)+x2  │                          Linear(512→2)
  └────────┬────────┘                                    │
  ┌────────▼────────┐                                    ▼
  │ Up(128→64)+x1   │                         cls_logits (B × 2)
  └────────┬────────┘                     [Authentic / Tampered]
           │
     Conv2d(64→1, 1×1)
           │
           ▼
  seg_logits (B × 1 × 256 × 256)
     [Per-pixel tamper mask]
```

**Key design choices:**
- **Shared encoder** — both heads benefit from the same learned features, and the classification signal acts as implicit regularization for the segmentation branch
- **DoubleConv block** — two sequential Conv2d → BatchNorm → ReLU layers forming the core building unit at every encoder/decoder stage
- **Skip connections** — each decoder stage concatenates upsampled features with the corresponding encoder features to recover spatial detail
- **Bottleneck classifier** — operates on the 1024-channel bottleneck via global average pooling, avoiding extra parameters from the full feature map

In [17]:
# ================== 4) Define the U-Net with classifier head ==================
class DoubleConv(nn.Module):
    """
    Purpose:
        Apply two convolutional layers with normalization and activation.

    Key Responsibilities:
        - Extract localized visual features.
        - Reuse the same feature block across encoder and decoder stages.

    Notes:
        This block is the core building unit of the U-Net backbone.
    """
    def __init__(self, in_ch, out_ch):
        """
        Purpose:
            Initialize the double-convolution block.

        Inputs:
            in_ch (int): Number of input channels.
            out_ch (int): Number of output channels.

        Returns:
            None.

        Notes:
            Batch normalization follows each convolution.
        """
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        """
        Purpose:
            Transform the input feature map with two convolution stages.

        Inputs:
            x (torch.Tensor): Input feature tensor.

        Returns:
            torch.Tensor: Feature tensor after convolution, normalization, and activation.

        Notes:
            This helper keeps encoder and decoder definitions concise.
        """
        return self.block(x)

class Down(nn.Module):
    """
    Purpose:
        Downsample feature maps in the encoder path.

    Key Responsibilities:
        - Reduce spatial resolution with max pooling.
        - Expand channel capacity with a DoubleConv block.

    Notes:
        This block helps the network capture higher-level tampering cues.
    """
    def __init__(self, in_ch, out_ch):
        """
        Purpose:
            Initialize one encoder stage.

        Inputs:
            in_ch (int): Number of incoming channels.
            out_ch (int): Number of outgoing channels.

        Returns:
            None.

        Notes:
            Pooling is applied before the convolution block.
        """
        super().__init__()
        self.pool = nn.MaxPool2d(2)
        self.conv = DoubleConv(in_ch, out_ch)

    def forward(self, x):
        """
        Purpose:
            Downsample the input feature map and refine it.

        Inputs:
            x (torch.Tensor): Encoder input feature map.

        Returns:
            torch.Tensor: Downsampled and refined feature map.

        Notes:
            The sequence preserves the original implementation exactly.
        """
        x = self.pool(x)
        x = self.conv(x)
        return x

class Up(nn.Module):
    """
    Purpose:
        Upsample decoder features and fuse them with encoder skip connections.

    Key Responsibilities:
        - Recover spatial detail in the decoder.
        - Merge coarse semantic features with fine-grained encoder activations.

    Notes:
        Padding compensates for minor shape mismatches before concatenation.
    """
    def __init__(self, in_channels, out_channels):
        """
        Purpose:
            Initialize one decoder stage.

        Inputs:
            in_channels (int): Number of channels entering the transposed convolution.
            out_channels (int): Number of channels produced after upsampling.

        Returns:
            None.

        Notes:
            The concatenated tensor is refined with DoubleConv.
        """
        super().__init__()
        self.up = nn.ConvTranspose2d(in_channels, out_channels, 2, stride=2)
        self.conv = DoubleConv(in_channels, out_channels)

    def forward(self, x1, x2):
        """
        Purpose:
            Upsample decoder features and concatenate them with encoder skip features.

        Inputs:
            x1 (torch.Tensor): Decoder feature map from the deeper level.
            x2 (torch.Tensor): Encoder skip connection from the matching spatial scale.

        Returns:
            torch.Tensor: Decoder feature map after skip fusion.

        Notes:
            Shape alignment is handled with explicit padding before concatenation.
        """
        x1 = self.up(x1)

        diffY = x2.size()[2] - x1.size()[2]
        diffX = x2.size()[3] - x1.size()[3]

        x1 = F.pad(x1, [diffX // 2, diffX - diffX // 2,
                        diffY // 2, diffY - diffY // 2])

        x = torch.cat([x2, x1], dim=1)
        return self.conv(x)

class UNetWithClassifier(nn.Module):
    """
    Purpose:
        Predict both image-level tampering labels and pixel-level tampering masks.

    Key Responsibilities:
        - Use a shared encoder-decoder backbone for localization.
        - Use a classifier head on the bottleneck to detect whether an image is tampered.

    Notes:
        The shared backbone lets the model learn complementary detection and localization signals.
    """
    def __init__(self, n_channels=3, n_classes=1, n_labels=2):
        """
        Purpose:
            Construct the shared U-Net backbone and the auxiliary classifier head.

        Inputs:
            n_channels (int): Number of image channels.
            n_classes (int): Number of segmentation output channels.
            n_labels (int): Number of classification labels.

        Returns:
            None.

        Notes:
            The architecture is preserved from the original notebook without structural changes.
        """
        super().__init__()

        # Encoder
        self.inc   = DoubleConv(n_channels, 64)
        self.down1 = Down(64, 128)
        self.down2 = Down(128, 256)
        self.down3 = Down(256, 512)
        self.down4 = Down(512, 1024)

        # Decoder
        self.up1 = Up(1024, 512)
        self.up2 = Up(512, 256)
        self.up3 = Up(256, 128)
        self.up4 = Up(128, 64)

        self.outc = nn.Conv2d(64, n_classes, kernel_size=1)

        # Classification head
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d((1, 1)),
            nn.Flatten(),
            nn.Linear(1024, 512),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),
            nn.Linear(512, n_labels),
        )

    def forward(self, x):
        """
        Purpose:
            Produce image-level logits and segmentation logits from the same input batch.

        Inputs:
            x (torch.Tensor): Batch of input images.

        Returns:
            tuple: Classification logits and segmentation logits.

        Notes:
            Classification uses the bottleneck representation, while segmentation uses the decoder output.
        """
        x1 = self.inc(x)
        x2 = self.down1(x1)
        x3 = self.down2(x2)
        x4 = self.down3(x3)
        x5 = self.down4(x4)   # bottleneck

        x = self.up1(x5, x4)
        x = self.up2(x,  x3)
        x = self.up3(x,  x2)
        x = self.up4(x,  x1)
        seg_logits = self.outc(x)

        cls_logits = self.classifier(x5)

        return cls_logits, seg_logits


In [18]:
model = UNetWithClassifier(
    n_channels=CONFIG['n_channels'],
    n_classes=CONFIG['n_classes'],
    n_labels=CONFIG['n_labels'],
).to(device)

# Wrap with DataParallel if multiple GPUs are available
if torch.cuda.device_count() > 1:
    model = nn.DataParallel(model)
    print(f'DataParallel enabled across {torch.cuda.device_count()} GPUs')


def get_base_model(m):
    """Unwrap DataParallel to access the underlying model."""
    return m.module if isinstance(m, nn.DataParallel) else m


total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Total parameters:     {total_params:,}')
print(f'Trainable parameters: {trainable_params:,}')

DataParallel enabled across 2 GPUs
Total parameters:     31,563,459
Trainable parameters: 31,563,459


## 8. Experiment Tracking

W&B is attached for experiment tracking.  Controlled by `CONFIG['use_wandb']`.
Falls back to offline mode if Kaggle secrets are unavailable.


In [19]:
import importlib.util
import subprocess

WANDB_ACTIVE = False
WANDB_RUN = None

if CONFIG['use_wandb']:
    WANDB_CONFIG = {k: v for k, v in CONFIG.items()}

    try:
        if importlib.util.find_spec("wandb") is None:
            subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "wandb"])

        import wandb

        try:
            from kaggle_secrets import UserSecretsClient
            wandb_api_key = UserSecretsClient().get_secret("WANDB_API_KEY")
            if not wandb_api_key:
                raise ValueError('WANDB_API_KEY is empty')
            wandb.login(key=wandb_api_key)
            WANDB_RUN = wandb.init(
                project="vK.10.6-tampered-image-detection-assignment",
                name=f"vK.10.6-unet-seed{SEED}-kaggle",
                tags=["vk10.6", "assignment", "amp", "checkpointing", "early-stopping", "multi-gpu", "eval-suite"],
                config=WANDB_CONFIG,
                reinit=True,
            )
            WANDB_ACTIVE = True
        except Exception as auth_exc:
            print(f"W&B online unavailable, switching to offline: {auth_exc}")
            WANDB_RUN = wandb.init(
                project="tampered-image-detection-assignment",
                name="vk10.6-offline",
                config=WANDB_CONFIG,
                mode="offline",
                reinit=True,
            )
            WANDB_ACTIVE = True
    except Exception as exc:
        print(f"W&B setup failed: {exc}")

print(f"W&B active: {WANDB_ACTIVE}")

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: harsh_vardhan (harsh_vardhan_iiitn) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: WARNING Using a boolean value for 'reinit' is deprecated. Use 'return_previous' or 'finish_previous' instead.
wandb: Tracking run with wandb version 0.24.0
wandb: Run data is saved locally in /kaggle/working/wandb/run-20260313_234123-1iigtw6a
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run vK.10.6-unet-seed42-kaggle
wandb: ⭐️ View project at https://wandb.ai/harsh_vardhan_iiitn/vK.10.6-tampered-image-detection-assignm

W&B active: True


## 9. Training Utilities

This section defines loss functions, evaluation metrics, checkpoint helpers,
and initializes the AMP scaler.  Loss functions are preserved verbatim from
the source notebook.


In [20]:
# ================== Loss functions, optimizer, scheduler, AMP scaler ==================

# Compute class weights from the training split
class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.array([0, 1]),
    y=train_df["label"].values,
)
class_weights = torch.tensor(class_weights, dtype=torch.float32).to(device)
print("Class weights:", class_weights)


class FocalLoss(nn.Module):
    """Focal-style classification loss that down-weights easy examples."""
    def __init__(self, alpha=None, gamma=2.0):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma

    def forward(self, logits, labels):
        ce = F.cross_entropy(logits, labels, weight=self.alpha, reduction='none')
        pt = torch.exp(-ce)
        focal = ((1 - pt) ** self.gamma) * ce
        return focal.mean()


def dice_loss(pred, target, eps=1e-7):
    """Soft Dice loss for segmentation logits."""
    prob = torch.sigmoid(pred)
    inter = (prob * target).sum(dim=(1,2,3))
    union = prob.sum(dim=(1,2,3)) + target.sum(dim=(1,2,3))
    dice = (2.0 * inter + eps) / (union + eps)
    return 1 - dice.mean()


criterion_cls = FocalLoss(alpha=class_weights, gamma=CONFIG['focal_gamma'])
bce_loss      = nn.BCEWithLogitsLoss()
ALPHA = CONFIG['alpha']
BETA  = CONFIG['beta']

optimizer = torch.optim.Adam(model.parameters(), lr=CONFIG['learning_rate'], weight_decay=CONFIG['weight_decay'])
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=CONFIG['scheduler_T_max'])

# AMP scaler
scaler = GradScaler('cuda', enabled=CONFIG['use_amp'])

print(f'Optimizer: Adam(lr={CONFIG["learning_rate"]}, wd={CONFIG["weight_decay"]})')
print(f'Scheduler: CosineAnnealingLR(T_max={CONFIG["scheduler_T_max"]})')
print(f'AMP: {"enabled" if CONFIG["use_amp"] else "disabled"}')
print(f'Loss weights: alpha={ALPHA}, beta={BETA}')

Class weights: tensor([0.8420, 1.2310], device='cuda:0')
Optimizer: Adam(lr=0.0001, wd=0.0001)
Scheduler: CosineAnnealingLR(T_max=50)
AMP: enabled
Loss weights: alpha=1.5, beta=1.0


### 9.1 Evaluation Metrics

Dice, IoU, and F1 are computed on thresholded binary masks.  To address metric
inflation from authentic images (where both prediction and ground truth are empty),
`compute_metrics_split()` reports metrics **separately for tampered-only samples**.


In [21]:
# ================== Evaluation Metrics ==================

def dice_coef(pred, target, eps=1e-7):
    """Dice coefficient for thresholded segmentation predictions."""
    prob = torch.sigmoid(pred)
    pred_bin = (prob > CONFIG['seg_threshold']).float()
    inter = (pred_bin * target).sum(dim=(1,2,3))
    union = pred_bin.sum(dim=(1,2,3)) + target.sum(dim=(1,2,3))
    dice = (2.0 * inter + eps) / (union + eps)
    return dice.mean().item()


def iou_coef(pred, target, eps=1e-7):
    """IoU for thresholded segmentation predictions."""
    prob = torch.sigmoid(pred)
    pred_bin = (prob > CONFIG['seg_threshold']).float()
    inter = (pred_bin * target).sum(dim=(1,2,3))
    union = pred_bin.sum(dim=(1,2,3)) + target.sum(dim=(1,2,3)) - inter
    return ((inter + eps) / (union + eps)).mean().item()


def f1_coef(pred, target, eps=1e-7):
    """Pixel-level F1 for thresholded segmentation predictions."""
    prob = torch.sigmoid(pred)
    pred_bin = (prob > CONFIG['seg_threshold']).float()
    tp = (pred_bin * target).sum(dim=(1,2,3))
    fp = (pred_bin * (1.0 - target)).sum(dim=(1,2,3))
    fn = ((1.0 - pred_bin) * target).sum(dim=(1,2,3))
    precision = (tp + eps) / (tp + fp + eps)
    recall = (tp + eps) / (tp + fn + eps)
    return (2.0 * precision * recall / (precision + recall + eps)).mean().item()


def compute_metrics_split(seg_logits, masks, labels):
    """Compute metrics for all samples and tampered-only samples separately."""
    all_dice, all_iou, all_f1 = [], [], []
    tam_dice, tam_iou, tam_f1 = [], [], []

    for i in range(seg_logits.size(0)):
        sl = seg_logits[i:i+1]
        m  = masks[i:i+1]
        d  = dice_coef(sl, m)
        io = iou_coef(sl, m)
        f  = f1_coef(sl, m)
        all_dice.append(d)
        all_iou.append(io)
        all_f1.append(f)

        if labels[i].item() == 1:  # tampered only
            tam_dice.append(d)
            tam_iou.append(io)
            tam_f1.append(f)

    return {
        'dice': np.mean(all_dice),
        'iou': np.mean(all_iou),
        'f1': np.mean(all_f1),
        'tampered_dice': np.mean(tam_dice) if tam_dice else 0.0,
        'tampered_iou': np.mean(tam_iou) if tam_iou else 0.0,
        'tampered_f1': np.mean(tam_f1) if tam_f1 else 0.0,
    }

### 9.2 Checkpoint Helpers

Three-file checkpoint strategy:
- `last_checkpoint.pt` — saved every epoch for seamless resume
- `best_model.pt` — saved when tampered-only Dice improves
- `checkpoint_epoch_N.pt` — periodic snapshot every N epochs


In [22]:
def save_checkpoint(state, filepath):
    """Save training state to a checkpoint file."""
    torch.save(state, filepath)


def load_checkpoint(filepath, model, optimizer, scaler, scheduler=None):
    """Restore training state from a checkpoint file."""
    ckpt = torch.load(filepath, map_location=device, weights_only=False)
    get_base_model(model).load_state_dict(ckpt['model_state_dict'])
    optimizer.load_state_dict(ckpt['optimizer_state_dict'])
    scaler.load_state_dict(ckpt['scaler_state_dict'])
    if scheduler is not None and 'scheduler_state_dict' in ckpt:
        scheduler.load_state_dict(ckpt['scheduler_state_dict'])
    restored_history = ckpt.get('history', None)
    return ckpt['epoch'] + 1, ckpt.get('best_metric', 0.0), ckpt.get('best_epoch', 0), restored_history


print('Checkpoint helpers defined.')

Checkpoint helpers defined.


## 10. Training Loop

The training loop uses:
- **AMP** for mixed precision training
- **Gradient clipping** at `max_grad_norm` for stability
- **Three-file checkpointing** with automatic resume
- **Early stopping** based on tampered-only Dice coefficient
- **Tampered-only metric tracking** for honest evaluation


In [23]:
def train_one_epoch(epoch):
    """Train for one epoch with AMP and gradient clipping. Returns loss, acc, and train dice."""
    model.train()
    running_loss = 0.0
    correct = 0
    all_seg_logits, all_masks, all_labels = [], [], []

    for images, masks, labels in tqdm(train_loader, desc=f'Train Ep{epoch}', leave=False):
        images = images.to(device)
        masks  = masks.to(device)
        labels = labels.to(device)

        optimizer.zero_grad(set_to_none=True)

        with autocast('cuda', enabled=CONFIG['use_amp']):
            cls_logits, seg_logits = model(images)
            loss_cls = criterion_cls(cls_logits, labels)
            loss_seg = CONFIG['seg_bce_weight'] * bce_loss(seg_logits, masks) + \
                       CONFIG['seg_dice_weight'] * dice_loss(seg_logits, masks)
            loss = ALPHA * loss_cls + BETA * loss_seg

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=CONFIG['max_grad_norm'])
        scaler.step(optimizer)
        scaler.update()

        running_loss += loss.item() * images.size(0)
        preds = torch.argmax(cls_logits, dim=1)
        correct += (preds == labels).sum().item()

        all_seg_logits.append(seg_logits.detach().cpu())
        all_masks.append(masks.cpu())
        all_labels.append(labels.cpu())

    scheduler.step()

    epoch_loss = running_loss / len(train_dataset)
    epoch_acc = correct / len(train_dataset)

    # Compute train segmentation dice (tampered-only)
    all_seg_logits = torch.cat(all_seg_logits)
    all_masks = torch.cat(all_masks)
    all_labels = torch.cat(all_labels)
    tampered_mask = all_labels == 1
    if tampered_mask.any():
        train_dice = dice_coef(all_seg_logits[tampered_mask], all_masks[tampered_mask])
    else:
        train_dice = 0.0

    return epoch_loss, epoch_acc, train_dice


In [24]:
@torch.no_grad()
def evaluate(loader, dataset_len, name='Val'):
    """Evaluate model with AMP, returning all-sample and tampered-only metrics."""
    model.eval()
    running_loss = 0.0
    correct = 0
    all_cls_logits, all_seg_logits, all_masks, all_labels = [], [], [], []

    for images, masks, labels in tqdm(loader, desc=name, leave=False):
        images = images.to(device)
        masks  = masks.to(device)
        labels = labels.to(device)

        with autocast('cuda', enabled=CONFIG['use_amp']):
            cls_logits, seg_logits = model(images)
            loss_cls = criterion_cls(cls_logits, labels)
            loss_seg = CONFIG['seg_bce_weight'] * bce_loss(seg_logits, masks) +                        CONFIG['seg_dice_weight'] * dice_loss(seg_logits, masks)
            loss = ALPHA * loss_cls + BETA * loss_seg

        running_loss += loss.item() * images.size(0)
        preds = torch.argmax(cls_logits, dim=1)
        correct += (preds == labels).sum().item()

        all_cls_logits.append(cls_logits.cpu())
        all_seg_logits.append(seg_logits.cpu())
        all_masks.append(masks.cpu())
        all_labels.append(labels.cpu())

    all_cls_logits = torch.cat(all_cls_logits)
    all_seg_logits = torch.cat(all_seg_logits)
    all_masks = torch.cat(all_masks)
    all_labels = torch.cat(all_labels)

    seg_metrics = compute_metrics_split(all_seg_logits, all_masks, all_labels)

    epoch_loss = running_loss / dataset_len
    epoch_acc = correct / dataset_len

    # ROC-AUC for classification
    probs = torch.softmax(all_cls_logits, dim=1)[:, 1].numpy()
    labels_np = all_labels.numpy()
    try:
        auc = roc_auc_score(labels_np, probs)
    except ValueError:
        auc = 0.0  # single class in batch

    seg_metrics['loss'] = epoch_loss
    seg_metrics['acc'] = epoch_acc
    seg_metrics['roc_auc'] = auc

    print(
        f'  {name} Loss: {epoch_loss:.4f} | Acc: {epoch_acc:.4f} | AUC: {auc:.4f} | '
        f'Dice(tam): {seg_metrics["tampered_dice"]:.4f} | '
        f'IoU(tam): {seg_metrics["tampered_iou"]:.4f}'
    )
    return seg_metrics


In [25]:
# ================== Training state initialization ==================
history = {
    "train_loss": [], "train_acc": [], "train_dice": [],
    "val_loss": [], "val_acc": [],
    "val_dice": [], "val_iou": [], "val_f1": [],
    "val_tampered_dice": [], "val_tampered_iou": [], "val_tampered_f1": [],
    "val_roc_auc": [],
    "lr": [],
}

best_metric = 0.0  # tampered-only Dice
best_epoch = 0
start_epoch = 1

# Resume from checkpoint if available
resume_path = os.path.join(CHECKPOINT_DIR, 'last_checkpoint.pt')
if os.path.exists(resume_path):
    start_epoch, best_metric, best_epoch, restored_history = load_checkpoint(
        resume_path, model, optimizer, scaler, scheduler
    )
    if restored_history:
        history = restored_history
    print(f'Resumed from epoch {start_epoch}, best tampered Dice={best_metric:.4f} at epoch {best_epoch}')
else:
    print('Starting fresh training.')


Starting fresh training.


In [26]:
# ================== Main training loop ==================
best_model_path = os.path.join(str(KAGGLE_WORKING_DIR), 'best_model.pth')

for epoch in range(start_epoch, CONFIG['max_epochs'] + 1):
    print(f'\nEpoch {epoch}/{CONFIG["max_epochs"]}')

    train_loss, train_acc, train_dice = train_one_epoch(epoch)
    print(f'  Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} | Train Dice(tam): {train_dice:.4f}')

    val_metrics = evaluate(val_loader, len(val_dataset), name='Val')

    # Record history
    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['train_dice'].append(train_dice)
    history['val_loss'].append(val_metrics['loss'])
    history['val_acc'].append(val_metrics['acc'])
    history['val_dice'].append(val_metrics['dice'])
    history['val_iou'].append(val_metrics['iou'])
    history['val_f1'].append(val_metrics['f1'])
    history['val_tampered_dice'].append(val_metrics['tampered_dice'])
    history['val_tampered_iou'].append(val_metrics['tampered_iou'])
    history['val_tampered_f1'].append(val_metrics['tampered_f1'])
    history['val_roc_auc'].append(val_metrics['roc_auc'])
    history['lr'].append(optimizer.param_groups[0]['lr'])

    # W&B logging
    if WANDB_ACTIVE:
        wandb.log({
            'epoch': epoch,
            'train/loss': train_loss, 'train/accuracy': train_acc, 'train/dice': train_dice,
            'val/loss': val_metrics['loss'], 'val/accuracy': val_metrics['acc'],
            'val/dice': val_metrics['dice'], 'val/iou': val_metrics['iou'],
            'val/f1': val_metrics['f1'],
            'val/tampered_dice': val_metrics['tampered_dice'],
            'val/tampered_iou': val_metrics['tampered_iou'],
            'val/tampered_f1': val_metrics['tampered_f1'],
            'val/roc_auc': val_metrics['roc_auc'],
            'lr': optimizer.param_groups[0]['lr'],
        })

    # Build checkpoint state (save unwrapped model for portability)
    state = {
        'epoch': epoch,
        'model_state_dict': get_base_model(model).state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scaler_state_dict': scaler.state_dict(),
        'scheduler_state_dict': scheduler.state_dict(),
        'best_metric': best_metric,
        'best_epoch': best_epoch,
        'config': CONFIG,
        'history': history,
    }

    # Save last checkpoint every epoch
    save_checkpoint(state, os.path.join(CHECKPOINT_DIR, 'last_checkpoint.pt'))

    # Save history CSV every epoch for crash resilience
    pd.DataFrame(history).to_csv(os.path.join(RESULTS_DIR, 'training_history.csv'), index=False)

    # Best model selection: tampered-only Dice
    current_metric = val_metrics['tampered_dice']
    if current_metric > best_metric:
        best_metric = current_metric
        best_epoch = epoch
        state['best_metric'] = best_metric
        state['best_epoch'] = best_epoch
        save_checkpoint(state, os.path.join(CHECKPOINT_DIR, 'best_model.pt'))
        torch.save(get_base_model(model).state_dict(), best_model_path)
        print(f'  => New best model (tampered Dice={best_metric:.4f})')

    # Periodic checkpoint
    if epoch % CONFIG['checkpoint_every'] == 0:
        save_checkpoint(state, os.path.join(CHECKPOINT_DIR, f'checkpoint_epoch_{epoch}.pt'))

    # Early stopping
    if epoch - best_epoch >= CONFIG['patience']:
        print(f'Early stopping at epoch {epoch}. Best tampered Dice={best_metric:.4f} at epoch {best_epoch}')
        break

print(f'\nTraining complete. Best tampered Dice={best_metric:.4f} at epoch {best_epoch}')
print(f'Training history saved to {RESULTS_DIR}/training_history.csv')


Epoch 1/100


Train Ep1:   0%|          | 0/275 [00:00<?, ?it/s]

  Train Loss: 0.9352 | Train Acc: 0.4738 | Train Dice(tam): 0.0111


Val:   0%|          | 0/60 [00:00<?, ?it/s]

  Val Loss: 0.8776 | Acc: 0.4059 | AUC: 0.6619 | Dice(tam): 0.0000 | IoU(tam): 0.0000
  => New best model (tampered Dice=0.0000)

Epoch 2/100


...

  Val Loss: 0.6992 | Acc: 0.8277 | AUC: 0.8981 | Dice(tam): 0.1811 | IoU(tam): 0.1316
  => New best model (tampered Dice=0.1811)

Epoch 100/100


Train Ep100:   0%|          | 0/275 [00:00<?, ?it/s]

  Train Loss: 0.7548 | Train Acc: 0.6858 | Train Dice(tam): 0.1088


Val:   0%|          | 0/60 [00:00<?, ?it/s]

  Val Loss: 0.7024 | Acc: 0.8224 | AUC: 0.8971 | Dice(tam): 0.1726 | IoU(tam): 0.1266

Training complete. Best tampered Dice=0.1811 at epoch 99
Training history saved to /kaggle/working/results/training_history.csv


## 11. Evaluation

Load the best checkpoint and evaluate on the held-out test split.
Reports both all-sample and tampered-only metrics.


In [27]:
# ================== Load best model and evaluate on test set ==================
best_ckpt_path = os.path.join(CHECKPOINT_DIR, 'best_model.pt')
if os.path.exists(best_ckpt_path):
    ckpt = torch.load(best_ckpt_path, map_location=device, weights_only=False)
    get_base_model(model).load_state_dict(ckpt['model_state_dict'])
    print(f'Loaded best model from epoch {ckpt["best_epoch"]}')
else:
    get_base_model(model).load_state_dict(torch.load(best_model_path, map_location=device))
    print('Loaded best model from legacy path')

test_metrics = evaluate(test_loader, len(test_dataset), name='Test')

print(f'\n{"="*50}')
print('FINAL TEST RESULTS')
print(f'{"="*50}')
print(f'Accuracy:         {test_metrics["acc"]:.4f}')
print(f'Dice (all):       {test_metrics["dice"]:.4f}')
print(f'IoU (all):        {test_metrics["iou"]:.4f}')
print(f'F1 (all):         {test_metrics["f1"]:.4f}')
print(f'Dice (tampered):  {test_metrics["tampered_dice"]:.4f}')
print(f'IoU (tampered):   {test_metrics["tampered_iou"]:.4f}')
print(f'F1 (tampered):    {test_metrics["tampered_f1"]:.4f}')
print(f'ROC-AUC:          {test_metrics["roc_auc"]:.4f}')

TRAINING_HISTORY = history
FINAL_TEST_METRICS = test_metrics

if WANDB_ACTIVE:
    wandb.summary.update({
        'best_epoch': best_epoch,
        'test/accuracy': test_metrics['acc'],
        'test/dice': test_metrics['dice'],
        'test/tampered_dice': test_metrics['tampered_dice'],
        'test/tampered_iou': test_metrics['tampered_iou'],
        'test/tampered_f1': test_metrics['tampered_f1'],
        'test/roc_auc': test_metrics['roc_auc'],
    })

Loaded best model from epoch 99


Test:   0%|          | 0/60 [00:00<?, ?it/s]

  Test Loss: 0.6823 | Acc: 0.8357 | AUC: 0.9057 | Dice(tam): 0.1946 | IoU(tam): 0.1418

FINAL TEST RESULTS
Accuracy:         0.8357
Dice (all):       0.4853
IoU (all):        0.4638
F1 (all):         0.4853
Dice (tampered):  0.1946
IoU (tampered):   0.1418
F1 (tampered):    0.1946
ROC-AUC:          0.9057


### 11.1 Metric Inflation Note

**Why tampered-only metrics matter:** Authentic images have all-zero ground truth masks.
A model that predicts all-zeros on authentic images gets perfect Dice/IoU for those samples,
inflating aggregate scores.  The tampered-only metrics isolate localization quality on images
that actually contain manipulated regions.


### 11.2 Training Curves

In [28]:
history_df = pd.DataFrame(history) if history['train_loss'] else pd.read_csv(
    os.path.join(RESULTS_DIR, 'training_history.csv'))

fig, axes = plt.subplots(2, 2, figsize=(16, 10))

epochs_range = history_df.index + 1

# Loss curves
axes[0,0].plot(epochs_range, history_df['train_loss'], label='Train Loss')
axes[0,0].plot(epochs_range, history_df['val_loss'], label='Val Loss')
axes[0,0].set_title('Loss Curves')
axes[0,0].set_xlabel('Epoch')
axes[0,0].legend()
axes[0,0].grid(True, alpha=0.3)

# Segmentation metrics (all + tampered-only)
if 'train_dice' in history_df.columns:
    axes[0,1].plot(epochs_range, history_df['train_dice'], label='Train Dice (tam)', ls='--', alpha=0.5)
axes[0,1].plot(epochs_range, history_df['val_dice'], label='Val Dice (all)', ls='--', alpha=0.5)
axes[0,1].plot(epochs_range, history_df['val_tampered_dice'], label='Dice (tampered)', lw=2)
axes[0,1].plot(epochs_range, history_df['val_tampered_iou'], label='IoU (tampered)')
axes[0,1].plot(epochs_range, history_df['val_tampered_f1'], label='F1 (tampered)')
axes[0,1].set_title('Segmentation Metrics')
axes[0,1].set_xlabel('Epoch')
axes[0,1].legend()
axes[0,1].grid(True, alpha=0.3)

# Accuracy
axes[1,0].plot(epochs_range, history_df['train_acc'], label='Train Acc')
axes[1,0].plot(epochs_range, history_df['val_acc'], label='Val Acc')
axes[1,0].set_title('Image-Level Accuracy')
axes[1,0].set_xlabel('Epoch')
axes[1,0].legend()
axes[1,0].grid(True, alpha=0.3)

# Learning rate
axes[1,1].plot(epochs_range, history_df['lr'], label='LR', color='orange')
axes[1,1].set_title('Learning Rate Schedule')
axes[1,1].set_xlabel('Epoch')
axes[1,1].legend()
axes[1,1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(PLOTS_DIR, 'training_curves.png'), dpi=150, bbox_inches='tight')
plt.show()

if WANDB_ACTIVE:
    wandb.log({'training_curves': wandb.Image(fig)})

### 11.3 Segmentation Threshold Optimization

Sweep the segmentation threshold from 0.05 to 0.80 on the **validation set** and select the threshold maximizing tampered-only F1. Then re-evaluate the test set with the optimal threshold.

In [29]:
# ================== Threshold Sweep on Validation Set ==================
@torch.no_grad()
def collect_predictions(loader):
    """Collect all predictions and ground truths from a dataloader."""
    model.eval()
    all_seg_logits, all_masks, all_labels, all_cls_logits = [], [], [], []
    for images, masks, labels in tqdm(loader, desc='Collecting predictions', leave=False):
        images = images.to(device)
        with autocast('cuda', enabled=CONFIG['use_amp']):
            cls_logits, seg_logits = model(images)
        all_seg_logits.append(seg_logits.cpu())
        all_masks.append(masks.cpu())
        all_labels.append(labels.cpu())
        all_cls_logits.append(cls_logits.cpu())
    return {
        'seg_logits': torch.cat(all_seg_logits),
        'masks': torch.cat(all_masks),
        'labels': torch.cat(all_labels),
        'cls_logits': torch.cat(all_cls_logits),
    }

val_preds = collect_predictions(val_loader)
test_preds = collect_predictions(test_loader)

thresholds = np.arange(0.05, 0.85, 0.05)
val_f1_scores = []
for thr in thresholds:
    pred_bin = (torch.sigmoid(val_preds['seg_logits']) > thr).float()
    tam_mask = val_preds['labels'] == 1
    if tam_mask.sum() == 0:
        val_f1_scores.append(0.0)
        continue
    tp = (pred_bin[tam_mask] * val_preds['masks'][tam_mask]).sum(dim=(1,2,3))
    fp = (pred_bin[tam_mask] * (1 - val_preds['masks'][tam_mask])).sum(dim=(1,2,3))
    fn = ((1 - pred_bin[tam_mask]) * val_preds['masks'][tam_mask]).sum(dim=(1,2,3))
    eps = 1e-7
    prec = (tp + eps) / (tp + fp + eps)
    rec = (tp + eps) / (tp + fn + eps)
    val_f1_scores.append((2 * prec * rec / (prec + rec + eps)).mean().item())

optimal_idx = np.argmax(val_f1_scores)
OPTIMAL_THRESHOLD = float(thresholds[optimal_idx])
print(f"Optimal threshold: {OPTIMAL_THRESHOLD:.2f} (val tampered F1 = {val_f1_scores[optimal_idx]:.4f})")

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(thresholds, val_f1_scores, 'b-o', markersize=4)
ax.axvline(OPTIMAL_THRESHOLD, color='r', linestyle='--', label=f'Optimal = {OPTIMAL_THRESHOLD:.2f}')
ax.axvline(0.5, color='gray', linestyle=':', alpha=0.5, label='Default = 0.50')
ax.set_xlabel('Threshold')
ax.set_ylabel('Tampered-Only F1')
ax.set_title('Segmentation Threshold Sweep (Validation Set)')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

def compute_metrics_at_threshold(seg_logits, masks, labels, threshold):
    eps = 1e-7
    results = {}
    for name, filt in [('all', None), ('tampered', labels == 1)]:
        sl = seg_logits if filt is None else seg_logits[filt]
        m = masks if filt is None else masks[filt]
        if len(sl) == 0:
            results[name] = {'dice': 0, 'iou': 0, 'f1': 0}
            continue
        pb = (torch.sigmoid(sl) > threshold).float()
        inter = (pb * m).sum(dim=(1,2,3))
        dice = ((2*inter+eps) / (pb.sum(dim=(1,2,3))+m.sum(dim=(1,2,3))+eps)).mean().item()
        iou = ((inter+eps) / (pb.sum(dim=(1,2,3))+m.sum(dim=(1,2,3))-inter+eps)).mean().item()
        tp = inter
        fp = (pb*(1-m)).sum(dim=(1,2,3))
        fn = ((1-pb)*m).sum(dim=(1,2,3))
        pr = (tp+eps)/(tp+fp+eps)
        rc = (tp+eps)/(tp+fn+eps)
        f1 = (2*pr*rc/(pr+rc+eps)).mean().item()
        results[name] = {'dice': dice, 'iou': iou, 'f1': f1}
    return results

test_default = compute_metrics_at_threshold(test_preds['seg_logits'], test_preds['masks'], test_preds['labels'], 0.5)
test_optimal = compute_metrics_at_threshold(test_preds['seg_logits'], test_preds['masks'], test_preds['labels'], OPTIMAL_THRESHOLD)

print(f"\n{'Metric':<25} {'Default (0.50)':>15} {'Optimal (' + f'{OPTIMAL_THRESHOLD:.2f}' + ')':>15} {'Delta':>10}")
print('-' * 68)
for metric in ['dice', 'iou', 'f1']:
    d = test_default['tampered'][metric]
    o = test_optimal['tampered'][metric]
    print(f"  Tampered {metric.upper():<15} {d:>15.4f} {o:>15.4f} {o-d:>+10.4f}")

Optimal threshold: 0.15 (val tampered F1 = 0.2104)

Metric                     Default (0.50)  Optimal (0.15)      Delta
--------------------------------------------------------------------
  Tampered DICE                     0.1946          0.2213    +0.0267
  Tampered IOU                      0.1418          0.1554    +0.0136
  Tampered F1                       0.1946          0.2213    +0.0267


### 11.4 Pixel-Level AUC-ROC

Threshold-independent localization quality metric.

In [30]:
# ================== Pixel-Level AUC-ROC ==================
from sklearn.metrics import roc_auc_score as sk_roc_auc_score

tam_mask = test_preds['labels'] == 1
tam_seg_probs = torch.sigmoid(test_preds['seg_logits'][tam_mask]).numpy().flatten()
tam_gt_masks = test_preds['masks'][tam_mask].numpy().flatten()
pixel_auc = sk_roc_auc_score(tam_gt_masks, tam_seg_probs)

cls_probs = torch.softmax(test_preds['cls_logits'], dim=1)[:, 1].numpy()
image_auc = sk_roc_auc_score(test_preds['labels'].numpy(), cls_probs)

print(f"Pixel-Level AUC-ROC (tampered only): {pixel_auc:.4f}")
print(f"Image-Level AUC-ROC (all samples):   {image_auc:.4f}")

Pixel-Level AUC-ROC (tampered only): 0.7083
Image-Level AUC-ROC (all samples):   0.9057


### 11.5 Classification Analysis: Confusion Matrix, ROC Curve, PR Curve

In [31]:
# ================== Confusion Matrix + ROC/PR Curves ==================
from sklearn.metrics import confusion_matrix, roc_curve, precision_recall_curve, average_precision_score
import seaborn as sns

cls_probs_test = torch.softmax(test_preds['cls_logits'], dim=1)[:, 1].numpy()
cls_preds_test = torch.argmax(test_preds['cls_logits'], dim=1).numpy()
labels_test = test_preds['labels'].numpy()

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
cm = confusion_matrix(labels_test, cls_preds_test)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=['Authentic', 'Tampered'], yticklabels=['Authentic', 'Tampered'])
axes[0].set_xlabel('Predicted')
axes[0].set_ylabel('Actual')
axes[0].set_title('Confusion Matrix (Image-Level)')

fpr, tpr, _ = roc_curve(labels_test, cls_probs_test)
axes[1].plot(fpr, tpr, 'b-', lw=2, label=f'AUC = {image_auc:.4f}')
axes[1].plot([0, 1], [0, 1], 'k--', alpha=0.3)
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].set_title('ROC Curve (Image-Level)')
axes[1].legend(loc='lower right')
axes[1].grid(True, alpha=0.3)

precision_arr, recall_arr, _ = precision_recall_curve(labels_test, cls_probs_test)
ap = average_precision_score(labels_test, cls_probs_test)
axes[2].plot(recall_arr, precision_arr, 'g-', lw=2, label=f'AP = {ap:.4f}')
axes[2].set_xlabel('Recall')
axes[2].set_ylabel('Precision')
axes[2].set_title('Precision-Recall Curve (Image-Level)')
axes[2].legend(loc='lower left')
axes[2].grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### 11.6 Per-Forgery-Type Evaluation

CASIA filenames: `Tp_D_*` = copy-move, `Tp_S_*` = splicing.

In [32]:
# ================== Per-Forgery-Type Evaluation ==================
thr = OPTIMAL_THRESHOLD
forgery_types = []
for path in test_df['image_path'].tolist():
    fname = os.path.basename(path)
    if fname.startswith('Tp_D'): forgery_types.append('copy-move')
    elif fname.startswith('Tp_S'): forgery_types.append('splicing')
    elif fname.startswith('Au'): forgery_types.append('authentic')
    else: forgery_types.append('unknown')
forgery_types = np.array(forgery_types)

print("Forgery type distribution in test set:")
for ft in ['authentic', 'splicing', 'copy-move']:
    print(f"  {ft}: {(forgery_types == ft).sum()}")

print(f"\nPer-type metrics (threshold={thr:.2f}):")
print(f"{'Type':<15} {'Count':>6} {'Dice':>8} {'IoU':>8} {'F1':>8}")
print('-' * 50)
seg_probs = torch.sigmoid(test_preds['seg_logits'])
for ft in ['splicing', 'copy-move']:
    ft_m = forgery_types == ft
    if ft_m.sum() == 0: continue
    pb = (seg_probs[ft_m] > thr).float()
    gt = test_preds['masks'][ft_m]
    eps = 1e-7
    inter = (pb*gt).sum(dim=(1,2,3))
    dice = ((2*inter+eps)/(pb.sum(dim=(1,2,3))+gt.sum(dim=(1,2,3))+eps)).mean().item()
    iou = ((inter+eps)/(pb.sum(dim=(1,2,3))+gt.sum(dim=(1,2,3))-inter+eps)).mean().item()
    tp = inter; fp = (pb*(1-gt)).sum(dim=(1,2,3)); fn = ((1-pb)*gt).sum(dim=(1,2,3))
    pr = (tp+eps)/(tp+fp+eps); rc = (tp+eps)/(tp+fn+eps)
    f1 = (2*pr*rc/(pr+rc+eps)).mean().item()
    print(f"  {ft:<13} {ft_m.sum():>6} {dice:>8.4f} {iou:>8.4f} {f1:>8.4f}")

Forgery type distribution in test set:
  authentic: 1124
  splicing: 509
  copy-move: 260

Per-type metrics (threshold=0.15):
Type             Count     Dice      IoU       F1
--------------------------------------------------
  splicing         509   0.1244   0.0785   0.1244
  copy-move        260   0.4110   0.3059   0.4110


### 11.7 Mask-Size Stratified Evaluation

Bucket by mask area: tiny (<2%), small (2-5%), medium (5-15%), large (>15%).

In [33]:
# ================== Mask-Size Stratified Evaluation ==================
thr = OPTIMAL_THRESHOLD
tam_indices = (test_preds['labels'] == 1).nonzero(as_tuple=True)[0]
mask_areas = np.array([test_preds['masks'][i].sum().item()/test_preds['masks'][i].numel()*100 for i in tam_indices])

buckets = [('tiny (<2%)',0,2),('small (2-5%)',2,5),('medium (5-15%)',5,15),('large (>15%)',15,100)]
print(f"Mask-size stratified evaluation (threshold={thr:.2f}):")
print(f"{'Bucket':<18} {'Count':>6} {'Dice':>8} {'F1':>8}")
print('-' * 44)
bucket_names, bucket_f1s, bucket_counts = [], [], []
for bname, lo, hi in buckets:
    sel = (mask_areas >= lo) & (mask_areas < hi)
    if sel.sum() == 0:
        print(f"  {bname:<16} {0:>6}      -        -")
        continue
    idx = tam_indices[sel]
    pb = (torch.sigmoid(test_preds['seg_logits'][idx]) > thr).float()
    gt = test_preds['masks'][idx]
    eps = 1e-7
    inter = (pb*gt).sum(dim=(1,2,3))
    dice = ((2*inter+eps)/(pb.sum(dim=(1,2,3))+gt.sum(dim=(1,2,3))+eps)).mean().item()
    tp = inter; fp = (pb*(1-gt)).sum(dim=(1,2,3)); fn = ((1-pb)*gt).sum(dim=(1,2,3))
    pr = (tp+eps)/(tp+fp+eps); rc = (tp+eps)/(tp+fn+eps)
    f1 = (2*pr*rc/(pr+rc+eps)).mean().item()
    print(f"  {bname:<16} {sel.sum():>6} {dice:>8.4f} {f1:>8.4f}")
    bucket_names.append(bname); bucket_f1s.append(f1); bucket_counts.append(int(sel.sum()))

if bucket_names:
    fig, ax = plt.subplots(figsize=(8, 4))
    bars = ax.bar(range(len(bucket_names)), bucket_f1s, color='steelblue')
    ax.set_xticks(range(len(bucket_names)))
    ax.set_xticklabels(bucket_names)
    ax.set_ylabel('Tampered-Only F1')
    ax.set_title('F1 by Mask Size Bucket')
    ax.set_ylim(0, 1)
    for bar, cnt in zip(bars, bucket_counts):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.02, f'n={cnt}', ha='center', fontsize=9)
    ax.grid(True, alpha=0.3, axis='y')
    plt.tight_layout()
    plt.show()

Mask-size stratified evaluation (threshold=0.15):
Bucket              Count     Dice       F1
--------------------------------------------
  tiny (<2%)          289   0.0972   0.0972
  small (2-5%)        190   0.2115   0.2115
  medium (5-15%)      171   0.3403   0.3403
  large (>15%)        119   0.3675   0.3675


### 11.8 Shortcut Learning Checks

1. **Mask randomization:** Shuffle GT masks, check F1 drop.
2. **Boundary sensitivity:** Erode predictions by 1px, check Dice stability.

In [34]:
# ================== Shortcut Learning Checks ==================
thr = OPTIMAL_THRESHOLD
tam_idx = (test_preds['labels'] == 1).nonzero(as_tuple=True)[0]
tam_seg = test_preds['seg_logits'][tam_idx]
tam_masks_sc = test_preds['masks'][tam_idx]
eps = 1e-7

def compute_tam_f1(pred_bin, gt_masks):
    tp = (pred_bin * gt_masks).sum(dim=(1,2,3))
    fp = (pred_bin * (1 - gt_masks)).sum(dim=(1,2,3))
    fn = ((1 - pred_bin) * gt_masks).sum(dim=(1,2,3))
    pr = (tp+eps)/(tp+fp+eps); rc = (tp+eps)/(tp+fn+eps)
    return (2*pr*rc/(pr+rc+eps)).mean().item()

pred_bin = (torch.sigmoid(tam_seg) > thr).float()
baseline_f1 = compute_tam_f1(pred_bin, tam_masks_sc)

shuffled_masks = tam_masks_sc[torch.randperm(len(tam_masks_sc))]
shuffled_f1 = compute_tam_f1(pred_bin, shuffled_masks)

eroded_pred = -F.max_pool2d(-pred_bin, kernel_size=3, stride=1, padding=1)
eroded_f1 = compute_tam_f1(eroded_pred, tam_masks_sc)

print("SHORTCUT LEARNING VALIDATION")
print("=" * 55)
print(f"  Baseline tampered F1:       {baseline_f1:.4f}")
print(f"  Shuffled-mask F1:           {shuffled_f1:.4f}  (delta: {shuffled_f1-baseline_f1:+.4f})")
print(f"  Eroded-prediction F1:       {eroded_f1:.4f}  (delta: {eroded_f1-baseline_f1:+.4f})")
print()
if baseline_f1 - shuffled_f1 > 0.05:
    print("  [PASS] Mask randomization: F1 drops -> model uses image content")
else:
    print("  [WARN] Mask randomization: F1 stable -> possible shortcut learning")
if abs(baseline_f1 - eroded_f1) < 0.10:
    print("  [PASS] Boundary sensitivity: F1 stable under erosion")
else:
    print("  [INFO] Boundary sensitivity: F1 changes with erosion")

SHORTCUT LEARNING VALIDATION
  Baseline tampered F1:       0.2213
  Shuffled-mask F1:           0.0658  (delta: -0.1555)
  Eroded-prediction F1:       0.2221  (delta: +0.0007)

  [PASS] Mask randomization: F1 drops -> model uses image content
  [PASS] Boundary sensitivity: F1 stable under erosion


## 12. Visualization of Predictions

The following cells collect authentic and tampered examples from the test loader and visualize model outputs.
For assignment submission, the notebook makes the qualitative evidence clearer by surfacing:

- original image
- ground-truth mask
- predicted mask
- overlay highlighting predicted manipulated regions

Together, these views show both detection and localization performance.

**Assignment alignment:** This section provides qualitative evidence that the model detects tampering and highlights the manipulated region.


In [35]:
def denormalize(img_tensor):
    """
    Purpose:
        Convert a normalized image tensor back to displayable RGB space.

    Inputs:
        img_tensor (torch.Tensor): Normalized image tensor in CHW format.

    Returns:
        torch.Tensor: Image tensor scaled back to the [0, 1] range for visualization.

    Notes:
        This helper reverses the ImageNet normalization used during preprocessing.
    """
    mean = torch.tensor([0.485, 0.456, 0.406]).view(3,1,1)
    std  = torch.tensor([0.229, 0.224, 0.225]).view(3,1,1)
    img = img_tensor.cpu() * std + mean
    img = torch.clamp(img, 0, 1)
    return img

In [36]:
get_base_model(model).load_state_dict(torch.load(best_model_path, map_location=device))
model.eval()
print('Best model loaded for visualization.')

Best model loaded for visualization.


### 12.1 Sample Collection for Qualitative Review

The next helper functions collect balanced authentic and tampered examples from the test split and convert predictions into reviewer-friendly visualizations.
This makes it easier to compare image-level decisions with the corresponding predicted tampering regions.

**Assignment alignment:** This subsection connects the evaluation outputs to qualitative evidence for both detection and localization.


In [37]:
def collect_samples(loader, num_real=5, num_fake=5):
    """
    Purpose:
        Gather a balanced set of authentic and tampered examples for qualitative visualization.

    Inputs:
        loader (DataLoader): Dataloader that provides images, masks, and image-level labels.
        num_real (int): Number of authentic examples to collect.
        num_fake (int): Number of tampered examples to collect.

    Returns:
        tuple: Two lists containing authentic samples and tampered samples.

    Notes:
        Each sample stores the input image, ground-truth mask, predicted mask probabilities, and image-level labels.
    """
    real_samples = []
    fake_samples = []

    with torch.no_grad():
        for images, masks, labels in loader:
            images = images.to(device)
            labels = labels.to(device)

            # Forward pass: compute both image-level and pixel-level predictions for qualitative review.
            cls_logits, seg_logits = model(images)
            seg_probs = torch.sigmoid(seg_logits)
            preds = torch.argmax(cls_logits, dim=1)

            for i in range(images.size(0)):
                sample = {
                    "image": images[i].cpu(),
                    "mask_true": masks[i].cpu(),
                    "mask_pred": seg_probs[i].cpu(),
                    "label": int(labels[i].item()),
                    "pred": int(preds[i].item())
                }

                if sample["label"] == 0 and len(real_samples) < num_real:
                    real_samples.append(sample)

                if sample["label"] == 1 and len(fake_samples) < num_fake:
                    fake_samples.append(sample)

                if len(real_samples) >= num_real and len(fake_samples) >= num_fake:
                    return real_samples, fake_samples

    return real_samples, fake_samples


real_samples, fake_samples = collect_samples(test_loader, 5, 5)

print("Collected Real:", len(real_samples), " Fake:", len(fake_samples))

Collected Real: 5  Fake: 5


In [38]:
import matplotlib.pyplot as plt
import numpy as np

def show_samples_with_masks(samples, title):
    """
    Purpose:
        Visualize predicted tampered regions as red overlays on top of the original image.

    Inputs:
        samples (list): Collection of sample dictionaries returned by collect_samples.
        title (str): Figure title shown above the panel.

    Returns:
        None.

    Notes:
        Overlay views help the reader see where the model believes manipulation evidence is concentrated.
    """
    n = len(samples)
    plt.figure(figsize=(4*n, 4))

    for i, s in enumerate(samples):
        img = denormalize(s["image"]).permute(1,2,0).numpy()
        # Threshold the predicted probability map to produce a binary tampering mask.
        mask_pred = (s["mask_pred"][0].numpy() > 0.5).astype(np.uint8)

        overlay = img.copy()
        # Color predicted tampered pixels in red so the localization output is easy to interpret.
        overlay[mask_pred==1] = [1, 0, 0]

        blended = (0.6 * img + 0.4 * overlay)

        plt.subplot(1, n, i+1)
        plt.imshow(blended)
        lbl = "Real" if s["label"]==0 else "Fake"
        pred_lbl = "Real" if s["pred"]==0 else "Fake"
        plt.title(f"{lbl} | Pred: {pred_lbl}")
        plt.axis("off")

    plt.suptitle(title, fontsize=16)
    plt.tight_layout()
    plt.show()


show_samples_with_masks(real_samples, "5 Real Images with Predicted Manipulation Regions")
show_samples_with_masks(fake_samples, "5 Fake Images with Predicted Manipulation Regions")

In [39]:
import matplotlib.pyplot as plt
import numpy as np

def show_image_and_mask(samples, title):
    """
    Purpose:
        Display each image together with its predicted binary tampering mask.

    Inputs:
        samples (list): Collection of sample dictionaries returned by collect_samples.
        title (str): Figure title shown above the grid.

    Returns:
        None.

    Notes:
        Each sample is shown in two rows: the original image first, then the thresholded predicted mask.
    """
    n = len(samples)
    plt.figure(figsize=(6*n, 6))

    for idx, s in enumerate(samples):
        # Convert the normalized tensor back to a displayable RGB image.
        img = denormalize(s["image"]).permute(1,2,0).numpy()

        # Threshold the predicted probabilities to obtain a black-and-white mask.
        mask_pred = (s["mask_pred"][0].numpy() > 0.5).astype(np.uint8)

        # Top row: original image with image-level ground truth and prediction labels.
        plt.subplot(2, n, idx+1)
        plt.imshow(img)
        lbl = "Real" if s["label"] == 0 else "Fake"
        pred_lbl = "Real" if s["pred"] == 0 else "Fake"
        plt.title(f"{lbl} | Pred: {pred_lbl}")
        plt.axis("off")

        # Bottom row: predicted binary mask for tampered-region localization.
        plt.subplot(2, n, n + idx + 1)
        plt.imshow(mask_pred, cmap="gray")
        plt.title("Predicted Mask")
        plt.axis("off")

    plt.suptitle(title, fontsize=18)
    plt.tight_layout()
    plt.show()

In [40]:
show_image_and_mask(real_samples, "5 Real Images + Predicted Masks")
show_image_and_mask(fake_samples, "5 Fake Images + Predicted Masks")


In [41]:
real_samples, fake_samples = collect_samples(test_loader, num_real=10, num_fake=10)
print("Collected Real:", len(real_samples), " Fake:", len(fake_samples))

import matplotlib.pyplot as plt
import numpy as np
import math

def show_image_and_mask(samples, title):
    """
    Purpose:
        Display image and predicted-mask pairs in a grid that limits each row to three samples.

    Inputs:
        samples (list): Collection of sample dictionaries returned by collect_samples.
        title (str): Figure title shown above the grid.

    Returns:
        None.

    Notes:
        Each sample occupies two neighboring columns: one for the image and one for the predicted mask.
    """
    total = len(samples)
    cols = 3
    rows = math.ceil(total / cols)

    plt.figure(figsize=(cols * 6, rows * 4))

    for idx, s in enumerate(samples):
        row = idx // cols
        col = idx % cols

        # Convert the normalized tensor back to RGB for display.
        img = denormalize(s["image"]).permute(1,2,0).numpy()

        # Threshold the predicted probabilities to form a binary tampering mask.
        mask_pred = (s["mask_pred"][0].numpy() > 0.5).astype(np.uint8)

        # Compute the first subplot column used by this sample inside the current row.
        base_col = col * 2

        img_pos  = row * cols * 2 + base_col + 1
        mask_pos = img_pos + 1

        # Show the original image and the image-level prediction.
        plt.subplot(rows, cols*2, img_pos)
        plt.imshow(img)
        lbl = "Real" if s["label"] == 0 else "Fake"
        pred_lbl = "Real" if s["pred"] == 0 else "Fake"
        plt.title(f"{lbl} | Pred: {pred_lbl}", fontsize=10)
        plt.axis("off")

        # Show the corresponding thresholded localization mask.
        plt.subplot(rows, cols*2, mask_pos)
        plt.imshow(mask_pred, cmap="gray")
        plt.title("Mask", fontsize=10)
        plt.axis("off")

    plt.suptitle(title, fontsize=18)
    plt.tight_layout()
    plt.show()

show_image_and_mask(real_samples, "10 Real Images (3 per row)")
show_image_and_mask(fake_samples, "10 Fake Images (3 per row)")

Collected Real: 10  Fake: 10


### 12.2 Submission-Ready Prediction Panels

The final visualization block packages the qualitative outputs into a compact four-panel format.
Each row aligns the original image, ground-truth mask, predicted mask, and overlay so the assignment reviewer can inspect localization quality quickly.

**Assignment alignment:** This subsection supports the final qualitative presentation requirement of the assignment.


In [42]:
import matplotlib.pyplot as plt
import numpy as np

def show_submission_prediction_grid(samples, title, max_items=6):
    """
    Purpose:
        Create a submission-style four-panel view for a small set of qualitative examples.

    Inputs:
        samples (list): Collection of sample dictionaries returned by collect_samples.
        title (str): Figure title shown above the grid.
        max_items (int): Maximum number of examples to visualize.

    Returns:
        matplotlib.figure.Figure | None: The generated figure when samples are available.

    Notes:
        Each row shows the original image, ground-truth mask, predicted mask, and overlay for side-by-side review.
    """
    chosen = samples[:max_items]
    if not chosen:
        print(f"No samples available for: {title}")
        return None

    fig, axes = plt.subplots(len(chosen), 4, figsize=(16, 4 * len(chosen)))
    if len(chosen) == 1:
        axes = np.expand_dims(axes, axis=0)

    column_titles = ["Original Image", "Ground Truth Mask", "Predicted Mask", "Overlay"]

    for row_idx, sample in enumerate(chosen):
        img = denormalize(sample["image"]).permute(1, 2, 0).numpy()
        gt_mask = (sample["mask_true"][0].numpy() > 0.5).astype(np.uint8)
        pred_mask = (sample["mask_pred"][0].numpy() > 0.5).astype(np.uint8)

        overlay = img.copy()
        # Highlight predicted tampered pixels in red to make the localization decision easy to inspect.
        overlay[pred_mask == 1] = [1, 0, 0]
        blended = 0.6 * img + 0.4 * overlay

        panels = [img, gt_mask, pred_mask, blended]
        cmaps = [None, "gray", "gray", None]

        for col_idx, panel in enumerate(panels):
            ax = axes[row_idx, col_idx]
            if cmaps[col_idx] is None:
                ax.imshow(panel)
            else:
                ax.imshow(panel, cmap=cmaps[col_idx])
            ax.axis("off")
            if row_idx == 0:
                ax.set_title(column_titles[col_idx])

        lbl = "Authentic" if sample["label"] == 0 else "Tampered"
        pred_lbl = "Authentic" if sample["pred"] == 0 else "Tampered"
        axes[row_idx, 0].set_ylabel(f"GT: {lbl}\nPred: {pred_lbl}", fontsize=10)

    plt.suptitle(title, fontsize=18)
    plt.tight_layout()
    plt.show()
    return fig

real_fig = show_submission_prediction_grid(real_samples, "Submission Grid: Authentic Image Predictions", max_items=4)
fake_fig = show_submission_prediction_grid(fake_samples, "Submission Grid: Tampered Image Predictions", max_items=4)

if WANDB_ACTIVE:
    if real_fig is not None:
        wandb.log({"submission_grid_authentic": wandb.Image(real_fig)})
    if fake_fig is not None:
        wandb.log({"submission_grid_tampered": wandb.Image(fake_fig)})

## 13. Advanced Analysis

### 13.1 Failure Case Analysis

Display the 10 worst predictions (lowest per-sample Dice) among tampered test images.

In [43]:
# ================== Failure Case Analysis ==================
thr = OPTIMAL_THRESHOLD
tam_idx_fc = (test_preds['labels'] == 1).nonzero(as_tuple=True)[0]

per_sample_dice = []
for i in tam_idx_fc:
    sl = test_preds['seg_logits'][i:i+1]
    m = test_preds['masks'][i:i+1]
    pb = (torch.sigmoid(sl) > thr).float()
    eps = 1e-7
    per_sample_dice.append(((2*(pb*m).sum()+eps)/(pb.sum()+m.sum()+eps)).item())
per_sample_dice = np.array(per_sample_dice)
worst_order = np.argsort(per_sample_dice)[:10]

fig, axes = plt.subplots(min(10, len(worst_order)), 4, figsize=(16, 4*min(10, len(worst_order))))
if len(worst_order) == 1: axes = axes[np.newaxis, :]
fig.suptitle('10 Worst Predictions (Lowest Dice)', fontsize=16, y=1.01)

for row, wi in enumerate(worst_order):
    gi = tam_idx_fc[wi].item()
    img_path = test_df.iloc[gi]['image_path']
    fname = os.path.basename(img_path)
    ftype = 'copy-move' if fname.startswith('Tp_D') else 'splicing' if fname.startswith('Tp_S') else 'unknown'
    mask_area = test_preds['masks'][gi].sum().item()/test_preds['masks'][gi].numel()*100
    orig = cv2.imread(img_path)
    if orig is not None:
        orig = cv2.cvtColor(cv2.resize(orig, (256,256)), cv2.COLOR_BGR2RGB)
    else:
        orig = np.zeros((256,256,3), dtype=np.uint8)
    gt = test_preds['masks'][gi,0].numpy()
    pred = (torch.sigmoid(test_preds['seg_logits'][gi,0]) > thr).float().numpy()
    overlay = orig.copy().astype(float)
    overlay[pred > 0.5] = overlay[pred > 0.5]*0.5 + np.array([255,0,0])*0.5
    axes[row,0].imshow(orig); axes[row,0].set_title(f'{ftype} | area={mask_area:.1f}%', fontsize=8)
    axes[row,1].imshow(gt, cmap='gray'); axes[row,1].set_title('Ground Truth', fontsize=8)
    axes[row,2].imshow(pred, cmap='gray'); axes[row,2].set_title(f'Pred (Dice={per_sample_dice[wi]:.3f})', fontsize=8)
    axes[row,3].imshow(overlay.astype(np.uint8)); axes[row,3].set_title('Overlay', fontsize=8)
    for ax in axes[row]: ax.axis('off')
plt.tight_layout()
plt.show()

## 14. Explainability

### 14.1 Grad-CAM Visualization

Grad-CAM heatmaps from the encoder bottleneck show what the model attends to.

In [44]:
# ================== Grad-CAM Explainability ==================
class GradCAM:
    def __init__(self, model, target_layer):
        self.model = model
        self.gradients = None
        self.activations = None
        target_layer.register_forward_hook(lambda m,i,o: setattr(self, 'activations', o.detach()))
        target_layer.register_full_backward_hook(lambda m,gi,go: setattr(self, 'gradients', go[0].detach()))

    def generate(self, input_tensor, target_class=None):
        self.model.eval()
        output = self.model(input_tensor)
        cls_logits = output[0] if isinstance(output, tuple) else output
        if target_class is None:
            target_class = cls_logits.argmax(dim=1).item()
        self.model.zero_grad()
        cls_logits[0, target_class].backward(retain_graph=True)
        weights = self.gradients.mean(dim=(2,3), keepdim=True)
        cam = F.relu((weights * self.activations).sum(dim=1, keepdim=True))
        cam = F.interpolate(cam, size=input_tensor.shape[2:], mode='bilinear', align_corners=False)
        cam = cam - cam.min()
        if cam.max() > 0: cam = cam / cam.max()
        return cam.squeeze().cpu().numpy()

base_model = get_base_model(model)
gradcam = GradCAM(base_model, base_model.down4.conv.block)

auth_idx = (test_preds['labels']==0).nonzero(as_tuple=True)[0][:3]
tamp_idx = (test_preds['labels']==1).nonzero(as_tuple=True)[0][:3]
sample_idx = torch.cat([auth_idx, tamp_idx])
labels_text = ['Authentic']*3 + ['Tampered']*3

fig, axes = plt.subplots(6, 3, figsize=(12, 24))
fig.suptitle('Grad-CAM: Encoder Attention Maps', fontsize=16, y=1.01)
for row, (idx, label) in enumerate(zip(sample_idx, labels_text)):
    img_path = test_df.iloc[idx.item()]['image_path']
    orig = cv2.imread(img_path)
    orig = cv2.cvtColor(cv2.resize(orig, (256,256)), cv2.COLOR_BGR2RGB) if orig is not None else np.zeros((256,256,3), dtype=np.uint8)
    inp = test_dataset[idx.item()][0].unsqueeze(0).to(device)
    cam = gradcam.generate(inp)
    heatmap = (plt.cm.jet(cam)[:,:,:3] * 255).astype(np.uint8)
    overlay = (orig*0.5 + heatmap*0.5).astype(np.uint8)
    with torch.no_grad():
        _, seg_out = base_model(inp)
    pred = (torch.sigmoid(seg_out[0,0]) > OPTIMAL_THRESHOLD).cpu().numpy()
    axes[row,0].imshow(orig); axes[row,0].set_title(f'Original ({label})', fontsize=9)
    axes[row,1].imshow(overlay); axes[row,1].set_title('Grad-CAM Overlay', fontsize=9)
    axes[row,2].imshow(pred, cmap='gray'); axes[row,2].set_title('Predicted Mask', fontsize=9)
    for ax in axes[row]: ax.axis('off')
plt.tight_layout()
plt.show()

## 15. Robustness Testing

Test the model against 8 post-processing degradation conditions at inference time.

In [45]:
# ================== Robustness Testing Suite ==================
def apply_jpeg(images, quality):
    result = []
    for img in images:
        arr = (img.permute(1,2,0).cpu().numpy()*255).clip(0,255).astype(np.uint8)
        _, enc = cv2.imencode('.jpg', cv2.cvtColor(arr, cv2.COLOR_RGB2BGR), [int(cv2.IMWRITE_JPEG_QUALITY), quality])
        dec = cv2.cvtColor(cv2.imdecode(enc, cv2.IMREAD_COLOR), cv2.COLOR_BGR2RGB).astype(np.float32)/255.0
        result.append(torch.from_numpy(dec).permute(2,0,1))
    return torch.stack(result)

def apply_noise(images, sigma):
    return (images + torch.randn_like(images)*(sigma/255.0)).clamp(0,1)

def apply_blur(images, ks):
    k = torch.ones(1,1,ks,ks,device=images.device)/(ks**2)
    return torch.cat([F.conv2d(images[:,c:c+1],k,padding=ks//2) for c in range(images.shape[1])], dim=1)

def apply_resize(images, scale):
    h,w = images.shape[2:]
    return F.interpolate(F.interpolate(images, scale_factor=scale, mode='bilinear', align_corners=False), size=(h,w), mode='bilinear', align_corners=False)

degradations = [
    ('JPEG QF=70', lambda x: apply_jpeg(x,70)),
    ('JPEG QF=50', lambda x: apply_jpeg(x,50)),
    ('Noise s=10', lambda x: apply_noise(x,10)),
    ('Noise s=25', lambda x: apply_noise(x,25)),
    ('Blur k=3',   lambda x: apply_blur(x,3)),
    ('Blur k=5',   lambda x: apply_blur(x,5)),
    ('Resize 0.75', lambda x: apply_resize(x,0.75)),
    ('Resize 0.50', lambda x: apply_resize(x,0.50)),
]

thr = OPTIMAL_THRESHOLD
clean_f1 = test_optimal['tampered']['f1']
robustness_results = []
model.eval()
mean_t = torch.tensor([0.485,0.456,0.406]).view(1,3,1,1)
std_t = torch.tensor([0.229,0.224,0.225]).view(1,3,1,1)

for deg_name, deg_fn in degradations:
    preds_list, gts_list = [], []
    for images, masks, labels in tqdm(test_loader, desc=deg_name, leave=False):
        imgs_01 = images * std_t + mean_t
        imgs_deg = deg_fn(imgs_01)
        imgs_norm = (imgs_deg - mean_t) / std_t
        tam = labels == 1
        if tam.sum() == 0: continue
        with torch.no_grad():
            _, sl = model(imgs_norm.to(device))
        preds_list.append((torch.sigmoid(sl.cpu()[tam]) > thr).float())
        gts_list.append(masks[tam])
    if not preds_list:
        robustness_results.append((deg_name, 0, 0))
        continue
    pb = torch.cat(preds_list); gt = torch.cat(gts_list)
    eps = 1e-7
    tp = (pb*gt).sum(dim=(1,2,3)); fp = (pb*(1-gt)).sum(dim=(1,2,3)); fn = ((1-pb)*gt).sum(dim=(1,2,3))
    f1 = (2*(tp+eps)/(tp+fp+eps)*(tp+eps)/(tp+fn+eps)/((tp+eps)/(tp+fp+eps)+(tp+eps)/(tp+fn+eps)+eps)).mean().item()
    robustness_results.append((deg_name, f1, f1-clean_f1))

print(f"\nROBUSTNESS RESULTS (threshold={thr:.2f})")
print(f"{'Condition':<18} {'F1':>8} {'Delta':>10}")
print('-' * 38)
print(f"  {'Clean':<16} {clean_f1:>8.4f} {'---':>10}")
for name, f1, delta in robustness_results:
    print(f"  {name:<16} {f1:>8.4f} {delta:>+10.4f}")

fig, ax = plt.subplots(figsize=(12, 5))
names = ['Clean'] + [r[0] for r in robustness_results]
f1s = [clean_f1] + [r[1] for r in robustness_results]
colors = ['green'] + ['steelblue']*len(robustness_results)
ax.bar(range(len(names)), f1s, color=colors)
ax.axhline(clean_f1, color='green', ls='--', alpha=0.5, label='Clean baseline')
ax.set_xticks(range(len(names)))
ax.set_xticklabels(names, rotation=30, ha='right', fontsize=9)
ax.set_ylabel('Tampered-Only F1')
ax.set_title('Robustness: F1 Under Post-Processing Degradations')
ax.legend()
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

JPEG QF=70:   0%|          | 0/60 [00:00<?, ?it/s]

JPEG QF=50:   0%|          | 0/60 [00:00<?, ?it/s]

Noise s=10:   0%|          | 0/60 [00:00<?, ?it/s]

Noise s=25:   0%|          | 0/60 [00:00<?, ?it/s]

Blur k=3:   0%|          | 0/60 [00:00<?, ?it/s]

Blur k=5:   0%|          | 0/60 [00:00<?, ?it/s]

Resize 0.75:   0%|          | 0/60 [00:00<?, ?it/s]

Resize 0.50:   0%|          | 0/60 [00:00<?, ?it/s]


ROBUSTNESS RESULTS (threshold=0.15)
Condition                F1      Delta
--------------------------------------
  Clean              0.2213        ---
  JPEG QF=70         0.1549    -0.0664
  JPEG QF=50         0.1682    -0.0531
  Noise s=10         0.1986    -0.0227
  Noise s=25         0.1728    -0.0485
  Blur k=3           0.2074    -0.0139
  Blur k=5           0.0748    -0.1465
  Resize 0.75        0.2132    -0.0081
  Resize 0.50        0.2062    -0.0151


## 16. Inference Examples

A self-contained inference function for running the trained model on
individual images after training.


In [46]:
def predict_single_image(image_path, model, device, threshold=None):
    """Run inference on a single image and return classification + segmentation mask."""
    if threshold is None:
        threshold = CONFIG['seg_threshold']

    image = cv2.imread(str(image_path))
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

    transform = get_valid_transform()
    augmented = transform(image=image, mask=np.zeros(image.shape[:2], dtype='float32'))
    img_tensor = augmented['image'].unsqueeze(0).to(device)

    model.eval()
    with torch.no_grad():
        with autocast('cuda', enabled=CONFIG['use_amp']):
            cls_logits, seg_logits = model(img_tensor)

    cls_pred = torch.argmax(cls_logits, dim=1).item()
    seg_prob = torch.sigmoid(seg_logits).cpu().squeeze()
    seg_mask = (seg_prob > threshold).numpy().astype(np.uint8)

    return {
        'classification': 'tampered' if cls_pred == 1 else 'authentic',
        'confidence': torch.softmax(cls_logits, dim=1).max().item(),
        'mask_probability': seg_prob.numpy(),
        'mask_binary': seg_mask,
    }

print('Inference function defined.')
print('Usage: result = predict_single_image("path/to/image.jpg", model, device)')

Inference function defined.
Usage: result = predict_single_image("path/to/image.jpg", model, device)


## 17. Conclusion

This notebook (vK.10.6) presents a complete, engineering-upgraded pipeline for tampered
image detection and localization.  All code runs in one notebook compatible with
Kaggle and Google Colab.

**New in vK.10.6 (evaluation suite):**
- Segmentation threshold optimization (sweep 0.05-0.80 on val set)
- Pixel-level AUC-ROC for threshold-independent localization quality
- Confusion matrix, ROC curve, and PR curve with AUC/AP annotations
- Per-forgery-type evaluation (splicing vs copy-move)
- Mask-size stratified evaluation (tiny/small/medium/large buckets)
- Shortcut learning validation (mask randomization + boundary sensitivity)
- Failure case analysis (10 worst predictions with metadata)
- Grad-CAM explainability on encoder bottleneck
- Robustness testing: 8 degradation conditions with delta from clean baseline
- Data leakage verification (path overlap assertion)

**Engineering improvements over vK.10.4:**
- Multi-GPU training with `nn.DataParallel` (utilizes both Kaggle T4 GPUs)
- Batch size auto-scaling based on total VRAM across all GPUs
- Portable checkpoints: always save/load the unwrapped model state_dict

**Carried forward from vK.10:**
- Removed duplicate prior experiment block (data leakage fix)
- Centralized CONFIG dictionary for all hyperparameters
- Full reproducibility seeding across Python, NumPy, and PyTorch
- Automatic Mixed Precision (AMP) for faster training
- Three-file checkpoint system with seamless resume
- Early stopping based on tampered-only Dice coefficient
- Tampered-only metric reporting to address metric inflation
- GPU diagnostics with VRAM-based batch size auto-adjustment
- Metadata caching to skip redundant dataset scanning
- Optimized DataLoaders with persistent workers and seeded sampling

**Assignment coverage:**
- Dataset: authentic images, tampered images, ground truth masks
- Model: image-level detection + pixel-level localization
- Evaluation: Dice, IoU, F1 (all-sample and tampered-only)
- Visualization: Original, Ground Truth, Predicted Mask, Overlay panels

In [47]:
if WANDB_ACTIVE and WANDB_RUN is not None:
    WANDB_RUN.finish()
    print('W&B run finished.')
else:
    print('No active W&B run to finish.')

wandb: updating run metadata
wandb: 
wandb: Run history:
wandb:          epoch ▁▁▁▂▂▂▂▃▃▃▃▄▄▄▄▄▄▄▅▅▅▅▅▅▆▆▆▆▆▇▇▇▇▇▇▇████
wandb:             lr ███▇▇▆▆▆▄▄▄▃▃▃▃▂▁▁▁▁▁▁▁▁▂▂▃▃▃▃▄▄▅▅▆▇▇███
wandb: train/accuracy ▁▁▁▁▂▂▃▃▃▄▅▆▆▇▇▇▇████████████▇▇▇▇▇█▇▇▇██
wandb:     train/dice ▁▁▁▁▁▂▃▄▄▄▅▆▇▇▇▇████▇███▇▇█▇▇▇▇▇▇▇▇▇█▇██
wandb:     train/loss █▄▄▃▃▃▃▃▃▃▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▂▂▁▂▂▁▂▁▁▁▁▁▁▁
wandb:   val/accuracy ▁▃▄▃▃▃▃▆▆▅▆▆▆▇▆▇▆▆▇▇▇▇▇▆▆▇▅▇▆▇▇▇▆▇██▆███
wandb:       val/dice ████████▇▅▂▅▂▃▁▃▁▃▃▂▂▂▂▂▁▁▁▂▃▁▃▄▅▂▂▂▅▅▂▆
wandb:         val/f1 █████▆▇▅▇▅▅▃▄▄▃▃▄▃▄▃▃▄▃▃▃▃▃▄▄▂▄▅▃▃▁▇▆▅▆▅
wandb:        val/iou █████▇█▇▄▇▇▃▅▃▄▄▄▃▃▄▄▃▃▃▃▄▄▅▆▅▃▂▅▁▂▄▆▅▄▅
wandb:       val/loss █▇▆▆▆▅▅▅▅▄▄▄▄▄▄▃▃▃▃▃▃▃▃▃▃▃▃▃▃▃▃▃▃▃▃▂▂▂▂▁
wandb:             +4 ...
wandb: 
wandb: Run summary:
wandb:         best_epoch 99
wandb:              epoch 100
wandb:                 lr 0.0001
wandb:      test/accuracy 0.83571
wandb:          test/dice 0.48528
wandb:       test/roc_auc 0.90568
wandb: test/tampered_dice 0.19457
wandb:   test/tampered_f1 0.194

W&B run finished.
